# Análise Exploratória

In [1]:
import sys
import numpy as np
import seaborn as sns
import pandas as pd
np.set_printoptions(threshold=sys.maxsize)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 80)
def id_like_dtypes(path):
    """Detecta colunas de identificação/código, CPF e CNPJ pelo nome e monta
    um dict de dtype=str para evitar perda de zeros à esquerda na leitura."""
    header = pd.read_csv(path, nrows=0).columns
    dtypes = {}
    for col in header:
        c = col.lower()
        is_id = (
            "codigo" in c or "cnpj" in c or "cpf" in c
            or c.startswith(("id", "numero", "sequencial", "cod"))
            or c.endswith(("id", "idpncp"))
            or c in {"nifornecedor"}
        )
        if is_id:
            dtypes[col] = str
    return dtypes


def read_csv_ids_str(path, dtype=None, **kwargs):
    """pd.read_csv que força colunas de identificação/código/CPF/CNPJ como string."""
    dtypes = id_like_dtypes(path)
    if dtype:
        dtypes.update(dtype)
    return pd.read_csv(path, dtype=dtypes, **kwargs)

In [2]:
raw_contratacoes = read_csv_ids_str('raw_contratacoes_202607071232.csv', low_memory=False)
raw_itens_contratacoes = read_csv_ids_str('raw_itens_contratacoes_202607071235.csv', low_memory=False)
raw_resultado_itens_contratacoes = read_csv_ids_str('raw_resultado_itens_contratacoes_202607071236.csv', low_memory=False)

# Domínio 4: Contratações (Lei 14.133/2021)

Editais e processos de compra publicados no PNCP (Portal Nacional de Contratações Públicas).


## `raw_contratacoes`
Cabeçalho do processo de contratação (edital).

| Coluna | Descrição |
|---|---|
| `idcompra` **PK** | Identificador único da compra no PNCP |
| `codigoorgao` **FK** → `raw_orgao` | Órgão comprador |
| `numerocontrolepncp` | Número de controle no PNCP |
| `objetocompra` | Descrição do objeto da contratação |
| `modalidadenome` | Pregão Eletrônico, Concorrência, Dispensa, etc. |
| `srp` | Indica se é Sistema de Registro de Preços (origem de ARP) |
| `valortotalestimado` | Valor estimado da contratação |
| `valortotalhomologado` | Valor efetivamente homologado |
| `datapublicacaopncp` | Data de publicação no PNCP |
| `dataaberturapropostapncp` | Data de abertura das propostas |
| `situacaocompranomepncp` | Situação atual (em andamento, encerrada, etc.) |
| `contratacaoexcluida` | Flag de exclusão lógica |

In [3]:
raw_contratacoes.shape

(112272, 53)

In [4]:
raw_contratacoes.duplicated().sum()

0

### `idcompra`
Identificador único da compra

In [5]:
raw_contratacoes.idcompra.duplicated().sum()

0

#### 112 mil compras

### `numerocontrolepncp`
Número de controle no PNCP

In [6]:
raw_contratacoes.numerocontrolepncp.duplicated().sum()

0

In [7]:
raw_contratacoes.head(3)

,idcompra,numerocontrolepncp,anocomprapncp,sequencialcomprapncp,orgaoentidadecnpj,orgaosubrogadocnpj,codigoorgao,orgaoentidaderazaosocial,orgaosubrogadorazaosocial,orgaoentidadeesferaid,orgaosubrogadoesferaid,orgaoentidadepoderid,orgaosubrogadopoderid,unidadeorgaocodigounidade,unidadesubrogadacodigounidade,unidadeorgaonomeunidade,unidadesubrogadanomeunidade,unidadeorgaoufsigla,unidadesubrogadaufsigla,unidadeorgaomunicipionome,unidadesubrogadamunicipionome,unidadeorgaocodigoibge,unidadesubrogadacodigoibge,numerocompra,modalidadeidpncp,codigomodalidade,modalidadenome,srp,mododisputaidpncp,codigomododisputa,amparolegalcodigopncp,amparolegalnome,amparolegaldescricao,informacaocomplementar,processo,objetocompra,existeresultado,orcamentosigilosocodigo,orcamentosigilosodescricao,situacaocompraidpncp,situacaocompranomepncp,tipoinstrumentoconvocatoriocodigopncp,tipoinstrumentoconvocatorionome,mododisputanomepncp,valortotalestimado,valortotalhomologado,datainclusaopncp,dataatualizacaopncp,datapublicacaopncp,dataaberturapropostapncp,dataencerramentopropostapncp,contratacaoexcluida,dt_ingest
0,16019506000422026,00394452000103-1-000561/2026,2026,561,00394452000103,NaN,44611,COMANDO DO EXERCITO,NaN,F,NaN,E,NaN,160195,NaN,COMANDO DO COMANDO MILITAR DO NORDESTE,NaN,PE,NaN,RECIFE,NaN,2611606,NaN,42,8,6,Dispensa,False,4,4,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que envolva valores inferiores a R$ ...,"O PMD 853110 constante do Termo de Referência está suspenso, razão pela qua...",64284.015347/2025-67,"Conjunto de Alarme Contra Intrusão – Tipo Sonoro, composto por: 1 Central de...",False,1,Compra sem sigilo,1,Divulgada no PNCP,2,Aviso de Contratação Direta,Dispensa Com Disputa,780.75,NaN,2026-01-02T09:04:43,2026-01-02T09:04:43,2026-01-02T09:04:43,2026-01-02T10:00:00,2026-01-08T10:00:00,False,2026-06-30T17:30:06.810413
1,38023206001242025,96291141000180-1-007957/2025,2025,7957,96291141000180,NaN,42434,SAO PAULO SECRETARIA DA ADMINISTRACAO PENITENCIARIA,NaN,E,NaN,E,NaN,380232,NaN,ESP-CTO DETENCAO PROVIS.TACIO AP.SANT DE CAIU,NaN,SP,NaN,CAIUÁ,NaN,3509106,NaN,124,8,6,Dispensa,False,5,5,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que envolva valores inferiores a R$ ...,NaN,006.00476090/2025-11,Aquisição de materiais para horta,True,1,Compra sem sigilo,1,Divulgada no PNCP,3,Ato que autoriza a Contratação Direta,Não se aplica,5569.20,5569.2,2026-01-02T09:25:43,2026-01-02T09:25:43,2026-01-02T09:25:43,NaN,NaN,False,2026-06-30T17:30:06.810413
2,92811806000502025,76206606000140-1-000642/2025,2025,642,76206606000140,NaN,38997,MUNICIPIO DE FOZ DO IGUACU,NaN,M,NaN,N,NaN,928118,NaN,FUNDAÇÃO MUNICIPAL DE SAÚDE DE FOZ DO IGUAÇU,NaN,PR,NaN,FOZ DO IGUAÇU,NaN,4108304,NaN,50,8,6,Dispensa,False,5,5,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que envolva valores inferiores a R$ ...,NaN,551/2025,"ENCADERNAÇÃO DE LIVROS CONTÁBEIS PARA O SETOR DE CONTABILIDADE, DO HOSPITAL ...",True,1,Compra sem sigilo,1,Divulgada no PNCP,3,Ato que autoriza a Contratação Direta,Não se aplica,165.00,165.0,2026-01-02T09:29:32,2026-01-02T09:29:32,2026-01-02T09:29:32,NaN,NaN,False,2026-06-30T17:30:06.810413


### `anocomprapncp`
Ano da compra no PNCP

In [8]:
raw_contratacoes.anocomprapncp.value_counts(1, dropna=False)

anocomprapncp
2026    0.919009
2025    0.072289
2027    0.008070
2024    0.000632
Name: proportion, dtype: float64

#### 91% das compras relativas ao ano de 2026

### `sequencialcomprapncp`
Sequencial da compra no ano no
PNCP

In [9]:
raw_contratacoes.sequencialcomprapncp.value_counts(dropna=False).head()

sequencialcomprapncp
1    1909
2    1817
3    1673
4    1585
5    1491
Name: count, dtype: int64

### `orgaoentidadecnpj`
CNPJ do órgão ou entidade responsável pela compra

In [10]:
raw_contratacoes.orgaoentidadecnpj.value_counts(1, dropna=False).head()

orgaoentidadecnpj
00394452000103    0.093425
00394502000144    0.048543
46374500000194    0.039752
96291141000180    0.032457
46377800000127    0.019996
Name: proportion, dtype: float64

#### Quase 10% das linhas são referentes ao comando do exército

In [11]:
raw_contratacoes['orgaoentidadecnpj'].str.len().value_counts()

orgaoentidadecnpj
14    112272
Name: count, dtype: int64

Todos cnpjs possuem 14 caracteres.

In [12]:
len(raw_contratacoes['orgaoentidadecnpj'].unique())

2882

#### 2882 cnpjs únicos nas contratações

### `orgaosubrogadocnpj`
CNPJ do órgão sub-rogado. Sub-rogado é aquele órgão que assumiu a posição de outro em relação a um direito ou obrigação.

In [13]:
raw_contratacoes.orgaosubrogadocnpj.value_counts(1, dropna=False).head()

orgaosubrogadocnpj
NaN               0.997444
00394502000144    0.000623
42414284000102    0.000579
10652179000115    0.000232
17217985000104    0.000223
Name: proportion, dtype: float64

#### Menos de 1% de preenchimento na coluna `orgaosubrogadocnpj`

### `orgaosubrogadoesferaid`
Identificador da esfera do órgão sub-rogado

In [14]:
raw_contratacoes.orgaosubrogadoesferaid.value_counts(0, dropna=False)

orgaosubrogadoesferaid
NaN    111985
F         260
E          19
M           7
N           1
Name: count, dtype: int64

In [15]:
raw_contratacoes[raw_contratacoes['orgaosubrogadoesferaid'] == "N"]

,idcompra,numerocontrolepncp,anocomprapncp,sequencialcomprapncp,orgaoentidadecnpj,orgaosubrogadocnpj,codigoorgao,orgaoentidaderazaosocial,orgaosubrogadorazaosocial,orgaoentidadeesferaid,orgaosubrogadoesferaid,orgaoentidadepoderid,orgaosubrogadopoderid,unidadeorgaocodigounidade,unidadesubrogadacodigounidade,unidadeorgaonomeunidade,unidadesubrogadanomeunidade,unidadeorgaoufsigla,unidadesubrogadaufsigla,unidadeorgaomunicipionome,unidadesubrogadamunicipionome,unidadeorgaocodigoibge,unidadesubrogadacodigoibge,numerocompra,modalidadeidpncp,codigomodalidade,modalidadenome,srp,mododisputaidpncp,codigomododisputa,amparolegalcodigopncp,amparolegalnome,amparolegaldescricao,informacaocomplementar,processo,objetocompra,existeresultado,orcamentosigilosocodigo,orcamentosigilosodescricao,situacaocompraidpncp,situacaocompranomepncp,tipoinstrumentoconvocatoriocodigopncp,tipoinstrumentoconvocatorionome,mododisputanomepncp,valortotalestimado,valortotalhomologado,datainclusaopncp,dataatualizacaopncp,datapublicacaopncp,dataaberturapropostapncp,dataencerramentopropostapncp,contratacaoexcluida,dt_ingest
71879,00060405900072026,18239038000187-1-000001/2026,2026,1,18239038000187,18239038000187,95740,EMPRESA DE INFORMATICA E INFORMACAO DO MUN.BHTE S/,EMPRESA DE INFORMATICA E INFORMACAO DO MUN.BHTE S/,N,N,N,N,0604,0604,EMPRESA DE INFORMÁTICA E INFORMAÇÃO DO MUNICÍPIO DE BELO HORIZONTE,EMPRESA DE INFORMÁTICA E INFORMAÇÃO DO MUNICÍPIO DE BELO HORIZONTE,MG,MG,BELO HORIZONTE,BELO HORIZONTE,3106200,3106200.0,90007,6,5,Pregão - Eletrônico,True,3,3,1,"Lei 14.133/2021, Art. 28, I",pregão: modalidade de licitação obrigatória para aquisição de bens e serviço...,O edital e seus anexos também estão disponíveis no site www.pbh.gov.br. Aler...,90007,"Promover o Registro de Preços, pelo período de 12 (doze) meses, para aquisiç...",True,1,Compra sem sigilo,1,Divulgada no PNCP,1,Edital,Aberto-Fechado,723228.8,151000.0,2026-01-14T07:10:18,2026-06-01T15:41:51,2026-01-14T07:10:18,2026-01-14T08:00:00,2026-01-27T09:00:00,False,2026-06-30T17:29:34.433174


In [16]:
raw_contratacoes[raw_contratacoes['orgaosubrogadoesferaid'] == "E"]

,idcompra,numerocontrolepncp,anocomprapncp,sequencialcomprapncp,orgaoentidadecnpj,orgaosubrogadocnpj,codigoorgao,orgaoentidaderazaosocial,orgaosubrogadorazaosocial,orgaoentidadeesferaid,orgaosubrogadoesferaid,orgaoentidadepoderid,orgaosubrogadopoderid,unidadeorgaocodigounidade,unidadesubrogadacodigounidade,unidadeorgaonomeunidade,unidadesubrogadanomeunidade,unidadeorgaoufsigla,unidadesubrogadaufsigla,unidadeorgaomunicipionome,unidadesubrogadamunicipionome,unidadeorgaocodigoibge,unidadesubrogadacodigoibge,numerocompra,modalidadeidpncp,codigomodalidade,modalidadenome,srp,mododisputaidpncp,codigomododisputa,amparolegalcodigopncp,amparolegalnome,amparolegaldescricao,informacaocomplementar,processo,objetocompra,existeresultado,orcamentosigilosocodigo,orcamentosigilosodescricao,situacaocompraidpncp,situacaocompranomepncp,tipoinstrumentoconvocatoriocodigopncp,tipoinstrumentoconvocatorionome,mododisputanomepncp,valortotalestimado,valortotalhomologado,datainclusaopncp,dataatualizacaopncp,datapublicacaopncp,dataaberturapropostapncp,dataencerramentopropostapncp,contratacaoexcluida,dt_ingest
1448,45386006006062025,03983541000175-1-000247/2025,2025,247,03983541000175,03464870000100,91742,MINISTERIO PUBLICO ESTADUAL,FUNDO ESPECIAL DE APOIO E DESENVOLVIMENTO DO MINISTERIO PUBLICO,E,E,J,E,453860,930216,MINISTÉRIO PÚBLICO DO MATO GROSSO DO SUL,FUNDO ESPECIAL APOIO E DESENV.MINIST.PÚBLICO,MS,MS,CAMPO GRANDE,CAMPO GRANDE,5002704,5002704.0,606,8,6,Dispensa,False,5,5,38,"Lei 14.133/2021, Art. 75, VIII","Dispensa de Licitação: nos casos de emergência ou de calamidade pública, qua...",NaN,09.2025.00014283-7,"Contratação de empresa para prestação de serviço de filmagem, gravação e tra...",True,1,Compra sem sigilo,1,Divulgada no PNCP,3,Ato que autoriza a Contratação Direta,Não se aplica,99240.00,99240.00,2026-01-12T18:06:08,2026-01-12T20:24:46,2026-01-12T18:06:08,NaN,NaN,False,2026-06-30T17:30:18.263374
4844,45386006006162026,03983541000175-1-000013/2026,2026,13,03983541000175,03464870000100,91742,MINISTERIO PUBLICO ESTADUAL,FUNDO ESPECIAL DE APOIO E DESENVOLVIMENTO DO MINISTERIO PUBLICO,E,E,J,E,453860,930216,MINISTÉRIO PÚBLICO DO MATO GROSSO DO SUL,FUNDO ESPECIAL APOIO E DESENV.MINIST.PÚBLICO,MS,MS,CAMPO GRANDE,CAMPO GRANDE,5002704,5002704.0,616,8,6,Dispensa,False,5,5,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que envolva valores inferiores a R$ ...,NaN,09.2026.00001067-4,"Serviços de decoração de ambientes, incluindo o fornecimento, montagem e des...",True,1,Compra sem sigilo,1,Divulgada no PNCP,3,Ato que autoriza a Contratação Direta,Não se aplica,4000.00,4000.00,2026-02-19T19:19:51,2026-02-19T19:29:23,2026-02-19T19:19:51,NaN,NaN,False,2026-06-30T17:30:24.045962
8900,45386006006152026,03983541000175-1-000015/2026,2026,15,03983541000175,03464870000100,91742,MINISTERIO PUBLICO ESTADUAL,FUNDO ESPECIAL DE APOIO E DESENVOLVIMENTO DO MINISTERIO PUBLICO,E,E,J,E,453860,930216,MINISTÉRIO PÚBLICO DO MATO GROSSO DO SUL,FUNDO ESPECIAL APOIO E DESENV.MINIST.PÚBLICO,MS,MS,CAMPO GRANDE,CAMPO GRANDE,5002704,5002704.0,615,8,6,Dispensa,False,5,5,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que envolva valores inferiores a R$ ...,NaN,09.2026.00001068-5,"Serviço de comunicação visual, produção, montagem e desmontagem de lona tipo...",True,1,Compra sem sigilo,1,Divulgada no PNCP,3,Ato que autoriza a Contratação Direta,Não se aplica,2380.00,2380.00,2026-02-20T16:11:56,2026-02-20T16:44:36,2026-02-20T16:11:56,NaN,NaN,False,2026-06-30T17:30:31.463012
9976,45386006006352026,03983541000175-1-000040/2026,2026,40,03983541000175,03464870000100,91742,MINISTERIO PUBLICO ESTADUAL,FUNDO ESPECIAL DE APOIO E DESENVOLVIMENTO DO MINISTERIO PUBLICO,E,E,J,E,453860,930216,MINISTÉRIO PÚBLICO DO MATO GROSSO DO SUL,FUNDO ESPECIAL APOIO E DESENV.MINIST.PÚBLICO,MS,MS,CAMPO GRANDE,CAMPO GRANDE,5002704,5002704.0,635,8,6,Dispensa,False,5,5,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que envolva valores infer

#### Apenas uma linha com `orgaosubrogadoesferaid` igal a `N`, possível sujeira

### `orgaosubrogadorazaosocial`
Razão social do órgão sub-rogado

In [17]:
raw_contratacoes.orgaosubrogadorazaosocial.value_counts(dropna=False)

orgaosubrogadorazaosocial
NaN                                                                           111985
COMANDO DA MARINHA                                                                70
COLEGIO PEDRO II                                                                  65
INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNOLOGIA DO PARANA                     26
UNIVERSIDADE FEDERAL DE MINAS GERAIS                                              25
FUNDO ESPECIAL DE APOIO E DESENVOLVIMENTO DO MINISTERIO PUBLICO                   19
INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNOLOGIA DO CEARA                      14
INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNOLOGIA DO TRIANGULO MINEIRO          11
INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNOLOGIA DE MATO GROSSO DO SUL         10
INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNOLOGIA DE PERNAMBUCO                  9
MINISTERIO DA CIDADANIA                                                            9
MUNICIPIO DE BELO HORIZONTE            

### `codigoorgao`
Código do órgão responsável

In [18]:
raw_contratacoes.codigoorgao.value_counts(1, dropna=False).head()

codigoorgao
44611    0.093425
46041    0.048543
34952    0.039752
42434    0.032457
35311    0.019996
Name: proportion, dtype: float64

### `orgaoentidaderazaosocial`
Razão social do órgão ou entidade

In [19]:
raw_contratacoes.orgaoentidaderazaosocial.value_counts(1, dropna=False).head(10)

orgaoentidaderazaosocial
COMANDO DO EXERCITO                                       0.093425
COMANDO DA MARINHA                                        0.048543
SECRETARIA DE ESTADO DA SAUDE                             0.041444
SAO PAULO SECRETARIA DA ADMINISTRACAO PENITENCIARIA       0.032457
SAO PAULO SECRETARIA DA SEGURANCA PUBLICA                 0.019996
EMPRESA BRASILEIRA DE SERVICOS HOSPITALARES - EBSERH      0.018295
UNIVERSIDADE DE SAO PAULO                                 0.017698
TRIBUNAL SUPERIOR DO TRABALHO                             0.013075
UNIVERSIDADE ESTADUAL PAULISTA JULIO DE MESQUITA FILHO    0.011739
COMANDO DA AERONAUTICA                                    0.011428
Name: proportion, dtype: float64

#### Top 10 órgãos nas contratacoes
| orgaoentidaderazaosocial                               |   proportion |
|:-------------------------------------------------------|-------------:|
| COMANDO DO EXERCITO                                    |         9,3 % |
| COMANDO DA MARINHA                                     |         4,9 % |
| SECRETARIA DE ESTADO DA SAUDE                          |         4,1 % |
| SAO PAULO SECRETARIA DA ADMINISTRACAO PENITENCIARIA    |         3,2 % |
| SAO PAULO SECRETARIA DA SEGURANCA PUBLICA              |         2 %    |
| EMPRESA BRASILEIRA DE SERVICOS HOSPITALARES - EBSERH   |         1,8 % |
| UNIVERSIDADE DE SAO PAULO                              |         1,8 % |
| TRIBUNAL SUPERIOR DO TRABALHO                          |         1,3 % |
| UNIVERSIDADE ESTADUAL PAULISTA JULIO DE MESQUITA FILHO |         1,2 % |
| COMANDO DA AERONAUTICA                                 |         1,1 % |

### `orgaoentidadeesferaid`
Esfera seria Federal, Municipal e Estadual. Poder seria Executivo, Legislativo e Judiciário.

In [20]:
raw_contratacoes.orgaoentidadeesferaid.value_counts(1, dropna=False)

orgaoentidadeesferaid
F    0.403921
E    0.284577
M    0.258800
N    0.045203
D    0.007500
Name: proportion, dtype: float64

#### (PERGUNTA) O que significam os valores `N` e `D` na coluna `orgaoentidadeesferaid`
Dicionário de valores. `D` parece ser referir ao Distrito Federal (Mas também inclui o `ESTADO DO AMAPA`, que parece se referir à Secretaria Extraordinária de Representação do Governo do Amapá em Brasília (SEAB), possível erro? apesar de o município da secretaria ser no DF é de responsabilidade do Amapá?).

In [21]:
raw_contratacoes[raw_contratacoes['orgaoentidadeesferaid'] == "N"].orgaoentidaderazaosocial.value_counts(dropna=False).head(10)

orgaoentidaderazaosocial
EMPRESA BRASILEIRA DE SERVICOS HOSPITALARES - EBSERH                             2054
INDUSTRIA DE MATERIAL BELICO DO BRASIL IMBEL                                      538
CONSORCIO INTERMUNICIPAL DE SAUDE ALTO DAS VERTENTES                              194
SERVICO DE APOIO AS MICRO E PEQ EMPRESAS DE SAO PAULO                             134
COMPANHIA AGUAS DE JOINVILLE                                                       80
URBANIZADORA MUNICIPAL SA URBAM                                                    73
FUNDACAO DE APOIO UNIVERSITARIO                                                    60
COMITE PARAOLIMPICO BRASILEIRO                                                     59
SERVICO SOCIAL DO COMERCIO - ADMINISTRACAO REGIONAL NO ESTADO DE MINAS GERAIS      56
SERVIÇO NACIONAL DE APRENDIZAGEM RURAL SENAR AR/MT                                 53
Name: count, dtype: int64

In [22]:
raw_contratacoes[raw_contratacoes['orgaoentidadeesferaid'] == "D"].orgaoentidaderazaosocial.value_counts(dropna=False)

orgaoentidaderazaosocial
DISTRITO FEDERAL SECRETARIA DE SAUDE                                                              226
DEPARTAMENTO DE ESTRADAS DE RODAGEM DO DISTRITO FEDERAL                                            64
CAMARA LEGISLATIVA DO DISTRITO FEDERAL                                                             62
SECRETARIA DE ESTADO DE ECONOMIA DO DISTRITO FEDERAL                                               59
POLICIA CIVIL DO DISTRITO FEDERAL                                                                  50
POLICIA MILITAR DO DISTRITO FEDERAL                                                                43
SECRETARIA DE ESTADO DE SEGURANCA PUBLICA DO DISTRITO FEDERAL                                      33
PROCURADORIA GERAL DO DISTRITO FEDERAL                                                             24
DEPARTAMENTO DE TRANSITO DO DISTRITO FEDERAL                                                       23
INSTITUTO DE ASSISTENCIA A SAUDE DOS SERVIDORES DO DISTRI

Para entender pq o valor `ESTADO DO AMAPA` parece na id de esfera `D`, que parece ser referir ao Distrito Federal, buscamos a linha com esse valor:

In [23]:
raw_contratacoes[raw_contratacoes['orgaoentidaderazaosocial'] == "ESTADO DO AMAPA"]

,idcompra,numerocontrolepncp,anocomprapncp,sequencialcomprapncp,orgaoentidadecnpj,orgaosubrogadocnpj,codigoorgao,orgaoentidaderazaosocial,orgaosubrogadorazaosocial,orgaoentidadeesferaid,orgaosubrogadoesferaid,orgaoentidadepoderid,orgaosubrogadopoderid,unidadeorgaocodigounidade,unidadesubrogadacodigounidade,unidadeorgaonomeunidade,unidadesubrogadanomeunidade,unidadeorgaoufsigla,unidadesubrogadaufsigla,unidadeorgaomunicipionome,unidadesubrogadamunicipionome,unidadeorgaocodigoibge,unidadesubrogadacodigoibge,numerocompra,modalidadeidpncp,codigomodalidade,modalidadenome,srp,mododisputaidpncp,codigomododisputa,amparolegalcodigopncp,amparolegalnome,amparolegaldescricao,informacaocomplementar,processo,objetocompra,existeresultado,orcamentosigilosocodigo,orcamentosigilosodescricao,situacaocompraidpncp,situacaocompranomepncp,tipoinstrumentoconvocatoriocodigopncp,tipoinstrumentoconvocatorionome,mododisputanomepncp,valortotalestimado,valortotalhomologado,datainclusaopncp,dataatualizacaopncp,datapublicacaopncp,dataaberturapropostapncp,dataencerramentopropostapncp,contratacaoexcluida,dt_ingest
35701,93005007000052026,00394577000206-1-000008/2026,2026,8,00394577000206,NaN,47069,ESTADO DO AMAPA,NaN,D,NaN,N,NaN,930050,NaN,EAP-SECRETARIA E. DE REPRE. G. AP EM BRASILIA,NaN,DF,NaN,BRASÍLIA,NaN,5300108,NaN,5,9,7,Inexigibilidade,False,5,5,50,"Lei 14.133/2021, Art. 74, caput",Inexigibilidade de Licitação: Todas as formas de inexigibilidade não previst...,NaN,SIGA 0048.06343.2301.0001/2025,CREDENCIAMENTO PARA FUTURA CONTRATAÇÃO DE EMPRESA ESPECIALIZADA PARA ELABORA...,True,1,Compra sem sigilo,1,Divulgada no PNCP,3,Ato que autoriza a Contratação Direta,Não se aplica,1194000.0,1194000.0,2026-05-25T15:56:42,2026-05-25T15:56:42,2026-05-25T15:56:42,NaN,NaN,False,2026-06-30T17:31:19.240678
74008,93005006000032026,00394577000206-1-000001/2026,2026,1,00394577000206,NaN,47069,ESTADO DO AMAPA,NaN,D,NaN,N,NaN,930050,NaN,EAP-SECRETARIA E. DE REPRE. G. AP EM BRASILIA,NaN,DF,NaN,BRASÍLIA,NaN,5300108,NaN,3,8,6,Dispensa,False,4,4,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que envolva valores inferiores a R$ ...,NaN,001.2026,"A aquisição de genêro alimentício café, destina-se ao atendimento das demand...",True,1,Compra sem sigilo,1,Divulgada no PNCP,2,Aviso de Contratação Direta,Dispensa Com Disputa,6720.0,6000.0,2026-02-11T14:31:26,2026-02-11T14:31:26,2026-02-11T14:31:26,2026-02-12T09:00:00,2026-02-20T09:00:00,False,2026-06-30T17:30:52.295132


### `orgaoentidadepoderid`
Identificador do poder do órgão
#### (PERGUNTA) Dicionário de valores?

In [24]:
raw_contratacoes.orgaoentidadepoderid.value_counts(1, dropna=False)

orgaoentidadepoderid
E    0.681550
N    0.224909
J    0.057583
L    0.035957
Name: proportion, dtype: float64

| orgaoentidadepoderid   |   proportion |
|:-----------------------|-------------:|
| E                      |           68 % |
| N                      |           22 % |
| J                      |            6 % |
| L                      |            4 % |

#### (PERGUNTA) O que significa o valor `N`? Parece se referir a Municípios.

In [25]:
raw_contratacoes[raw_contratacoes['orgaoentidadepoderid'] == "N"].head(10)

,idcompra,numerocontrolepncp,anocomprapncp,sequencialcomprapncp,orgaoentidadecnpj,orgaosubrogadocnpj,codigoorgao,orgaoentidaderazaosocial,orgaosubrogadorazaosocial,orgaoentidadeesferaid,orgaosubrogadoesferaid,orgaoentidadepoderid,orgaosubrogadopoderid,unidadeorgaocodigounidade,unidadesubrogadacodigounidade,unidadeorgaonomeunidade,unidadesubrogadanomeunidade,unidadeorgaoufsigla,unidadesubrogadaufsigla,unidadeorgaomunicipionome,unidadesubrogadamunicipionome,unidadeorgaocodigoibge,unidadesubrogadacodigoibge,numerocompra,modalidadeidpncp,codigomodalidade,modalidadenome,srp,mododisputaidpncp,codigomododisputa,amparolegalcodigopncp,amparolegalnome,amparolegaldescricao,informacaocomplementar,processo,objetocompra,existeresultado,orcamentosigilosocodigo,orcamentosigilosodescricao,situacaocompraidpncp,situacaocompranomepncp,tipoinstrumentoconvocatoriocodigopncp,tipoinstrumentoconvocatorionome,mododisputanomepncp,valortotalestimado,valortotalhomologado,datainclusaopncp,dataatualizacaopncp,datapublicacaopncp,dataaberturapropostapncp,dataencerramentopropostapncp,contratacaoexcluida,dt_ingest
2,92811806000502025,76206606000140-1-000642/2025,2025,642,76206606000140,NaN,38997,MUNICIPIO DE FOZ DO IGUACU,NaN,M,NaN,N,NaN,928118,NaN,FUNDAÇÃO MUNICIPAL DE SAÚDE DE FOZ DO IGUAÇU,NaN,PR,NaN,FOZ DO IGUAÇU,NaN,4108304,NaN,50,8,6,Dispensa,False,5,5,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que envolva valores inferiores a R$ ...,NaN,551/2025,"ENCADERNAÇÃO DE LIVROS CONTÁBEIS PARA O SETOR DE CONTABILIDADE, DO HOSPITAL ...",True,1,Compra sem sigilo,1,Divulgada no PNCP,3,Ato que autoriza a Contratação Direta,Não se aplica,165.00,165.00,2026-01-02T09:29:32,2026-01-02T09:29:32,2026-01-02T09:29:32,NaN,NaN,False,2026-06-30T17:30:06.810413
13,98600106009582025,42498733000148-1-002564/2025,2025,2564,42498733000148,NaN,34319,MUNICIPIO DE RIO DE JANEIRO,NaN,M,NaN,N,NaN,986001,NaN,PREFEITURA MUNICIPAL DO RIO DE JANEIRO - RJ,NaN,RJ,NaN,RIO DE JANEIRO,NaN,3304557,NaN,958,8,6,Dispensa,False,5,5,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que envolva valores inferiores a R$ ...,NaN,TRA-PRO-2025/00629,contratação de empresa especializada para o desenvolvimento de pesquisa e pr...,True,1,Compra sem sigilo,1,Divulgada no PNCP,3,Ato que autoriza a Contratação Direta,Não se aplica,30000.00,30000.00,2026-01-02T11:24:45,2026-01-02T11:24:45,2026-01-02T11:24:45,NaN,NaN,False,2026-06-30T17:30:06.810413
17,98600106001472026,42498733000148-1-000103/2026,2026,103,42498733000148,NaN,34319,MUNICIPIO DE RIO DE JANEIRO,NaN,M,NaN,N,NaN,986001,NaN,PREFEITURA MUNICIPAL DO RIO DE JANEIRO - RJ,NaN,RJ,NaN,RIO DE JANEIRO,NaN,3304557,NaN,147,8,6,Dispensa,False,5,5,18,"Lei 14.133/2021, Art. 75, I",Dispensa de Licitação: para contratação que envolva valores inferiores a R$ ...,NaN,TRA-PRO-2025/00736,Prestação de serviço técnicos especializados para elaboração de projeto de\n...,True,1,Compra sem sigilo,1,Divulgada no PNCP,3,Ato que autoriza a Contratação Direta,Não se aplica,110000.00,110000.00,2026-01-02T11:46:04,2026-01-02T11:46:04,2026-01-02T11:46:04,NaN,NaN,False,2026-06-30T17:30:06.810413
19,92811806000512025,76206606000140-1-000643/2025,2025,643,76206606000140,NaN,38997,MUNICIPIO DE FOZ DO IGUACU,NaN,M,NaN,N,NaN,928118,NaN,FUNDAÇÃO MUNICIPAL DE SAÚDE DE FOZ DO IGUAÇU,NaN,PR,NaN,FOZ DO IGUAÇU,NaN,4108304,NaN,51,8,6,Dispensa,False,5,5,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que envolva valores inferiores a R$ ...,NaN,584/2025,"AQUISIÇÃO DE INSTRUMENTAIS PARA OTORRINOLARINGOLOGIA, PARA O SETOR DE CENTRO...",True,1,Compra sem sigilo,1,Divulgada no PNCP,3,Ato que autoriza a Contratação Direta,Não se aplica,1244875.00,19980.00,2026-01-02T12:05:17,2026-01-02T12:05:17,2026-01-02T12:05:17,NaN,NaN,False,2026-06-30T17:30:06.810413
20,36500106000012026,33749086000290-1-000001/2026,2026,1,33749086000290,NaN,96919,FINANCIADORA DE ESTUDOS E PROJETOS - FINEP,NaN,N,NaN,N,NaN,365001,NaN,FINANCIADORA DE ESTUDOS 

In [26]:
raw_contratacoes[raw_contratacoes['orgaoentidadepoderid'] == "L"].head(10)

,idcompra,numerocontrolepncp,anocomprapncp,sequencialcomprapncp,orgaoentidadecnpj,orgaosubrogadocnpj,codigoorgao,orgaoentidaderazaosocial,orgaosubrogadorazaosocial,orgaoentidadeesferaid,orgaosubrogadoesferaid,orgaoentidadepoderid,orgaosubrogadopoderid,unidadeorgaocodigounidade,unidadesubrogadacodigounidade,unidadeorgaonomeunidade,unidadesubrogadanomeunidade,unidadeorgaoufsigla,unidadesubrogadaufsigla,unidadeorgaomunicipionome,unidadesubrogadamunicipionome,unidadeorgaocodigoibge,unidadesubrogadacodigoibge,numerocompra,modalidadeidpncp,codigomodalidade,modalidadenome,srp,mododisputaidpncp,codigomododisputa,amparolegalcodigopncp,amparolegalnome,amparolegaldescricao,informacaocomplementar,processo,objetocompra,existeresultado,orcamentosigilosocodigo,orcamentosigilosodescricao,situacaocompraidpncp,situacaocompranomepncp,tipoinstrumentoconvocatoriocodigopncp,tipoinstrumentoconvocatorionome,mododisputanomepncp,valortotalestimado,valortotalhomologado,datainclusaopncp,dataatualizacaopncp,datapublicacaopncp,dataaberturapropostapncp,dataencerramentopropostapncp,contratacaoexcluida,dt_ingest
15,92546506000032026,27532498000190-1-000003/2026,2026,3,27532498000190,NaN,76128,TRIBUNAL DE CONTAS DO MUNICIPIO DO RIO DE JANEIRO,NaN,M,NaN,L,NaN,925465,NaN,TRIBUNAL DE CONTAS DO MUNICIPIO DO RJ,NaN,RJ,NaN,RIO DE JANEIRO,NaN,3304557,NaN,3,8,6,Dispensa,False,4,4,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que envolva valores inferiores a R$ ...,"As condições, quantidades e exigências estão estabelecidas no Aviso de Contr...",40/103620/2025,Aquisição de Jalecos para uso dos profissionais de saúde do CMU a serem con...,False,1,Compra sem sigilo,1,Divulgada no PNCP,2,Aviso de Contratação Direta,Dispensa Com Disputa,5586.00,NaN,2026-01-02T11:43:24,2026-01-02T11:43:24,2026-01-02T11:43:24,2026-01-02T12:00:00,2026-01-07T08:00:00,False,2026-06-30T17:30:06.810413
151,01000106000132026,00530352000159-1-000008/2026,2026,8,00530352000159,NaN,47563,CAMARA DOS DEPUTADOS,NaN,F,NaN,L,NaN,010001,NaN,CAMARA DOS DEPUTADOS,NaN,DF,NaN,BRASÍLIA,NaN,5300108,NaN,13,8,6,Dispensa,False,4,4,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que envolva valores inferiores a R$ ...,NaN,1219612/2025,Contratação de empresa para prestação de serviço de PROCESSAMENTO E ESTERILI...,True,1,Compra sem sigilo,1,Divulgada no PNCP,2,Aviso de Contratação Direta,Dispensa Com Disputa,28475.18,26800.00,2026-01-05T15:36:14,2026-01-05T15:36:14,2026-01-05T15:36:14,2026-01-05T16:00:00,2026-01-13T08:00:00,False,2026-06-30T17:30:06.810413
163,92704506000742025,34870246000136-1-000094/2025,2025,94,34870246000136,NaN,80941,TRIBUNAL DE CONTAS DO ESTADO DO AMAPA,NaN,E,NaN,L,NaN,927045,NaN,TRIBUNAL DE CONTAS DO ESTADO DO AMAPÁ,NaN,AP,NaN,MACAPÁ,NaN,1600303,NaN,74,8,6,Dispensa,False,5,5,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que envolva valores inferiores a R$ ...,NaN,012678/2025,Contratação de link dedicado de internet de 500 Mbps para o evento “VIII TCE...,True,1,Compra sem sigilo,1,Divulgada no PNCP,3,Ato que autoriza a Contratação Direta,Não se aplica,4500.00,4500.00,2026-01-06T09:45:37,2026-01-06T09:45:37,2026-01-06T09:45:37,NaN,NaN,False,2026-06-30T17:30:06.810413
177,92942506000022026,51642635000123-1-000001/2026,2026,1,51642635000123,NaN,37420,CAMARA MUNICIPAL DE CUBATAO,NaN,M,NaN,L,NaN,929425,NaN,CAMARA MUNICIPAL DE CUBATAO - SP,NaN,SP,NaN,CUBATÃO,NaN,3513504,NaN,2,8,6,Dispensa,False,4,4,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que envolva valores inferiores a R$ ...,NaN,39/2025,CÂMERA FOTOGRÁFICA DIGITAL\nSensor Full Frame ExmorR BSI 33MP;\nEstabilizaçã...,True,1,Compra sem sigilo,1,Divulgada no PNCP,2,Aviso de Contratação Direta,Dispensa Com Disputa,25252.57,20928.98,2026-01-05T17:34:42,2026-01-05T17:34:42,2026-01-05T17:34:42,2026-01-07T08:00:00,2026-01-13T09:00:00,False,2026-06-30T17:30:06.810413
198,92704506000782025,34870246000136-1-000091/2025,2025,91,34870246000136,NaN,

#### Por algum motivo o valor `L` inclui tribunal de contas além das camaras municipais e a federal.

### `orgaosubrogadopoderid`
Identificador do poder do órgão sub-rogado

In [27]:
raw_contratacoes.orgaosubrogadopoderid.value_counts(1, dropna=False)

orgaosubrogadopoderid
NaN    0.997444
E      0.002485
N      0.000071
Name: proportion, dtype: float64

### `unidadeorgaocodigounidade`
Código da unidade do órgão

In [28]:
raw_contratacoes.unidadeorgaocodigounidade.value_counts(dropna=False).tail()

unidadeorgaocodigounidade
01612865000171-001    1
928937                1
929035                1
928859                1
981142                1
Name: count, dtype: int64

In [29]:
raw_contratacoes['unidadeorgaocodigounidade'].str.len().value_counts()

unidadeorgaocodigounidade
6     112258
1          4
7          3
3          2
14         2
5          1
4          1
18         1
Name: count, dtype: int64

#### O padrão parece ser de 6 numero, mas 14  unidades fogem desse padrão.

In [30]:
len(raw_contratacoes['unidadeorgaocodigounidade'].unique())

5369

### 5369 unidades distintas

### `unidadesubrogadacodigounidade`
Código da unidade subrogada

In [31]:
raw_contratacoes.unidadesubrogadacodigounidade.value_counts(dropna=False).head()

unidadesubrogadacodigounidade
NaN       111985
782802        33
765700        20
930216        19
155634        11
Name: count, dtype: int64

### `unidadeorgaonomeunidade`
Nome da unidade do órgão

In [32]:
raw_contratacoes.unidadeorgaonomeunidade.value_counts(1, dropna=False).head()

unidadeorgaonomeunidade
ESP-UNIVERSIDADE ESTADUAL DE CAMPINAS           0.009851
PREFEITURA MUNICIPAL DO RIO DE JANEIRO - RJ     0.009424
PMSP - SECRETARIA MUNICIPAL DE CULTURA/SP       0.007972
GOVERNO DO ESTADO DO CEARÁ                      0.007954
ESP-INST. ASSIST. MEDICA SERV. PUB. ESTADUAL    0.006751
Name: proportion, dtype: float64

### `unidadesubrogadanomeunidade`
Nome da unidade subrogada

In [33]:
raw_contratacoes.unidadesubrogadanomeunidade.value_counts(1, dropna=False).head()

unidadesubrogadanomeunidade
NaN                                             0.997444
CENTRO DE INTENDENCIA DA MARINHA EM SALVADOR    0.000294
CENTRO MEDICO ASSISTENCIAL DA MARINHA           0.000178
FUNDO ESPECIAL APOIO E DESENV.MINIST.PÚBLICO    0.000169
COLEGIO PEDRO II/CAMPUS TIJUCA II               0.000098
Name: proportion, dtype: float64

### `unidadeorgaoufsigla`
Sigla da unidade federativa da unidade do órgão

In [34]:
raw_contratacoes.unidadeorgaoufsigla.value_counts(1, dropna=False).head()

unidadeorgaoufsigla
SP    0.287498
RJ    0.119237
PR    0.078265
DF    0.065813
MG    0.054911
Name: proportion, dtype: float64

### `unidadesubrogadaufsigla`
Sigla da unidade federativa da unidade subrogada

In [35]:
raw_contratacoes.unidadesubrogadaufsigla.value_counts(1, dropna=False).head()

unidadesubrogadaufsigla
NaN    0.997444
RJ     0.000891
MG     0.000419
BA     0.000294
MS     0.000258
Name: proportion, dtype: float64

### `unidadeorgaomunicipionome`
Nome do município da unidade do órgão

In [36]:
raw_contratacoes.unidadeorgaomunicipionome.value_counts(1, dropna=False).head()

unidadeorgaomunicipionome
SÃO PAULO         0.130139
RIO DE JANEIRO    0.073536
BRASÍLIA          0.065813
FORTALEZA         0.023309
CURITIBA          0.022953
Name: proportion, dtype: float64

### `unidadesubrogadamunicipionome`
Nome do município da unidade subrogada

In [37]:
raw_contratacoes.unidadesubrogadamunicipionome.value_counts(1, dropna=False).head()

unidadesubrogadamunicipionome
NaN               0.997444
RIO DE JANEIRO    0.000793
BELO HORIZONTE    0.000294
SALVADOR          0.000294
CAMPO GRANDE      0.000205
Name: proportion, dtype: float64

### `unidadeorgaocodigoibge`
Código IBGE do município da unidade do órgão

In [38]:
raw_contratacoes.unidadeorgaocodigoibge.value_counts(1, dropna=False).head()

unidadeorgaocodigoibge
3550308    0.128972
3304557    0.072939
5300108    0.065413
2304400    0.023256
4106902    0.022650
Name: proportion, dtype: float64

### `unidadesubrogadacodigoibge`
Código IBGE do município da unidade subrogada

In [39]:
raw_contratacoes.unidadesubrogadacodigoibge.value_counts(1, dropna=False).head()

unidadesubrogadacodigoibge
NaN          0.997444
3304557.0    0.000793
3106200.0    0.000294
2927408.0    0.000294
5002704.0    0.000205
Name: proportion, dtype: float64

### `numerocompra`
#### (PERGUNTA) Qual a diferença entre `idcompra` e o `numerocompra`?

In [40]:
raw_contratacoes.numerocompra.value_counts(dropna=False).head()

numerocompra
90001    3451
90002    2668
90003    2224
90004    1855
90005    1626
Name: count, dtype: int64

In [41]:
raw_contratacoes[raw_contratacoes['numerocompra'] == "90001"].head(3)

,idcompra,numerocontrolepncp,anocomprapncp,sequencialcomprapncp,orgaoentidadecnpj,orgaosubrogadocnpj,codigoorgao,orgaoentidaderazaosocial,orgaosubrogadorazaosocial,orgaoentidadeesferaid,orgaosubrogadoesferaid,orgaoentidadepoderid,orgaosubrogadopoderid,unidadeorgaocodigounidade,unidadesubrogadacodigounidade,unidadeorgaonomeunidade,unidadesubrogadanomeunidade,unidadeorgaoufsigla,unidadesubrogadaufsigla,unidadeorgaomunicipionome,unidadesubrogadamunicipionome,unidadeorgaocodigoibge,unidadesubrogadacodigoibge,numerocompra,modalidadeidpncp,codigomodalidade,modalidadenome,srp,mododisputaidpncp,codigomododisputa,amparolegalcodigopncp,amparolegalnome,amparolegaldescricao,informacaocomplementar,processo,objetocompra,existeresultado,orcamentosigilosocodigo,orcamentosigilosodescricao,situacaocompraidpncp,situacaocompranomepncp,tipoinstrumentoconvocatoriocodigopncp,tipoinstrumentoconvocatorionome,mododisputanomepncp,valortotalestimado,valortotalhomologado,datainclusaopncp,dataatualizacaopncp,datapublicacaopncp,dataaberturapropostapncp,dataencerramentopropostapncp,contratacaoexcluida,dt_ingest
555,93365503900012026,01541156000142-1-000006/2026,2026,6,01541156000142,NaN,48528,JANDAIA DO SUL CAMARA MUNICIPAL,NaN,M,NaN,L,NaN,933655,NaN,MPR-CÂMARA MUNICIPAL DE JANDAIA DO SUL,NaN,PR,NaN,JANDAIA DO SUL,NaN,4112108,NaN,90001,4,3,Concorrência - Eletrônica,False,1,1,2,"Lei 14.133/2021, Art. 28, II",concorrência: modalidade de licitação para contratação de bens e serviços es...,Pasta tecnica disponível em tps://jandaiadosul.oxy.elotech.com.br/portaltran...,01,Execução de obra de reforma do prédio da nova sede da Câmara Municipal de Ja...,True,1,Compra sem sigilo,1,Divulgada no PNCP,1,Edital,Aberto,1778401.3,1100500.0,2026-04-02T07:01:12,2026-04-02T07:01:12,2026-04-02T07:01:12,2026-04-02T08:00:00,2026-04-23T09:30:00,False,2026-06-30T17:28:01.888366
922,27507805900012026,42357483000126-1-000001/2026,2026,1,42357483000126,NaN,95172,COMPANHIA BRASILEIRA DE TRENS URBANOS,NaN,F,NaN,E,NaN,275078,NaN,SUPERINTENDENCIA DE TRENS URBANOS DE MACEIÓ,NaN,AL,NaN,MACEIÓ,NaN,2704302,NaN,90001,6,5,Pregão - Eletrônico,True,1,1,1,"Lei 14.133/2021, Art. 28, I",pregão: modalidade de licitação obrigatória para aquisição de bens e serviço...,Para as respostas de esclarecimentos e impugnações deste edital acesse o li...,004/2026,Aquisição de baterias para VLT's,False,1,Compra sem sigilo,2,Revogada,1,Edital,Aberto,0.0,NaN,2026-02-04T07:11:10,2026-03-12T07:28:29,2026-02-04T07:11:10,2026-02-04T08:00:00,2026-02-20T10:00:00,False,2026-06-30T17:29:53.356012
2797,99000405900012026,08574719000148-1-000001/2026,2026,1,08574719000148,NaN,58320,SECRETARIA DE TURISMO,NaN,E,NaN,E,NaN,990004,NaN,ESP-SECRETARIA DE TURISMO E VIAGENS,NaN,SP,NaN,SÃO PAULO,NaN,3550308,NaN,90001,6,5,Pregão - Eletrônico,True,1,1,1,"Lei 14.133/2021, Art. 28, I",pregão: modalidade de licitação obrigatória para aquisição de bens e serviço...,"Considerando ausência de código especifico para cada item no comprasgov, sol...",02700000094202601,Constituição de ata de registro de prelos para fornecimento de infraestrutur...,True,1,Compra sem sigilo,1,Divulgada no PNCP,1,Edital,Aberto,71203805.9,42677500.0,2026-03-03T07:24:57,2026-03-03T07:24:57,2026-03-03T07:24:57,2026-03-03T08:00:00,2026-03-17T10:00:00,False,2026-06-30T17:29:52.436203


In [42]:
len(raw_contratacoes.numerocompra.unique())

3762

#### Existem apenas 3.7k números de compra, comparado a 112 mil ids de compra

### `modalidadeidpncp`
Identificador da modalidade de compra no PNCP

In [43]:
raw_contratacoes.modalidadeidpncp.value_counts(dropna=False)

modalidadeidpncp
8    42319
6    39579
9    25958
4     4160
5      184
7       72
Name: count, dtype: int64

### `codigomodalidade`
Código da modalidade de compra
#### (PERGUNTA) Qual a diferença entre o código da modalidade e o id da modalidade?

In [44]:
raw_contratacoes.codigomodalidade.value_counts(dropna=False)

codigomodalidade
6    42319
5    39651
7    25958
3     4344
Name: count, dtype: int64

### `modalidadenome`
Nome da modalidade de compra

In [45]:
raw_contratacoes.modalidadenome.value_counts(1, dropna=False)

modalidadenome
Dispensa                     0.376933
Pregão - Eletrônico          0.352528
Inexigibilidade              0.231206
Concorrência - Eletrônica    0.037053
Concorrência - Presencial    0.001639
Pregão - Presencial          0.000641
Name: proportion, dtype: float64

#### Top modalidades de contratação
| `modalidadenome`             |   Proporção |
|:--------------------------|-------------:|
| Dispensa                  |         37,7 % |
| Pregão - Eletrônico       |         35,3 % |
| Inexigibilidade           |         23,1 % |
| Concorrência - Eletrônica |          3,7 % |
| Concorrência - Presencial |          0,2 % |
| Pregão - Presencial       |          0,1 % |

### `srp`
Indicador se a compra é por Sistema
de Registro de Preços (SRP)
0 – False/Não
1 – True/Sim

In [46]:
raw_contratacoes.srp.value_counts(1, dropna=False)

srp
False    0.830305
True     0.169695
Name: proportion, dtype: float64

#### 17% das contratações feitas pelo SRP.

### `mododisputaidpncp`
Identificador do modo de disputa no PNCP

In [47]:
raw_contratacoes.mododisputaidpncp.value_counts(1, dropna=False)

mododisputaidpncp
5    0.395557
1    0.241191
4    0.212600
3    0.145254
2    0.003091
6    0.002307
Name: proportion, dtype: float64

### `codigomododisputa`
Código do modo de disputa
#### `mododisputaidpncp` e `codigomododisputa` parecem ser iguais
#### (PERGUNTA) Existe dicionário de valores?

In [48]:
raw_contratacoes.mododisputaidpncp.value_counts(1, dropna=False)

mododisputaidpncp
5    0.395557
1    0.241191
4    0.212600
3    0.145254
2    0.003091
6    0.002307
Name: proportion, dtype: float64

In [49]:
(raw_contratacoes.mododisputaidpncp != raw_contratacoes.mododisputaidpncp).sum()

0

### `amparolegalcodigopncp`
Código do amparo legal no PNCP

In [50]:
raw_contratacoes.amparolegalcodigopncp.value_counts(1, dropna=False).head()

amparolegalcodigopncp
1     0.352724
19    0.312447
16    0.070258
13    0.048391
50    0.040580
Name: proportion, dtype: float64

### `amparolegalnome`
Nome do amparo legal

In [51]:
raw_contratacoes.amparolegalnome.value_counts(1, dropna=False).head()

amparolegalnome
Lei 14.133/2021, Art. 28, I         0.352724
Lei 14.133/2021, Art. 75, II        0.312447
Lei 14.133/2021, Art. 74, IV        0.070258
Lei 14.133/2021, Art. 74, III, f    0.048391
Lei 14.133/2021, Art. 74, caput     0.040580
Name: proportion, dtype: float64

#### Top Artigos de amparo legal
| amparolegalnome                  |   Proporção |
|:---------------------------------|-------------:|
| Lei 14.133/2021, Art. 28, I      |         35,3 % |
| Lei 14.133/2021, Art. 75, II     |         31,2 % |
| Lei 14.133/2021, Art. 74, IV     |          7 %   |
| Lei 14.133/2021, Art. 74, III, f |          4,8 % |
| Lei 14.133/2021, Art. 74, caput  |          4,1 % |
| Lei 14.133/2021, Art. 28, II     |          3,9 % |
| Lei 14.133/2021, Art. 74, I      |          3,8 % |
| Lei 14.133/2021, Art. 74, II     |          1,6 % |
| Lei 14.133/2021, Art. 75, VIII   |          1,6 % |
| Lei 14.133/2021, Art. 75, I      |          1,2 % |

### `amparolegaldescricao`
Descrição do amparo legal

In [52]:
raw_contratacoes.amparolegaldescricao.value_counts(1, dropna=False).head()

amparolegaldescricao
pregão: modalidade de licitação obrigatória para aquisição de bens e serviços comuns                                                                 0.352724
Dispensa de Licitação: para contratação que envolva valores inferiores a R$ 50.000,00 (cinquenta mil reais), no caso de outros serviços e compras    0.312447
Inexigibilidade de Licitação: objetos que devam ou possam ser contratados por meio de credenciamento                                                 0.070258
Inexigibilidade de Licitação: f) treinamento e aperfeiçoamento de pessoal                                                                            0.048391
Inexigibilidade de Licitação: Todas as formas de inexigibilidade não previstas no art. 74                                                            0.040580
Name: proportion, dtype: float64

### `informacaocomplementar`
Informações complementares sobre a compra

In [53]:
raw_contratacoes.informacaocomplementar.value_counts(0, dropna=False).head()

informacaocomplementar
NaN                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

#### 45% dos valores sem informação complementar

### `processo`
Número do processo administrativo
relacionado

In [54]:
raw_contratacoes.processo.value_counts(0, dropna=False).head()

processo
64318.029552/2024-94     564
64305.033801/2024-95     468
64318.025437/2024-41     461
64278.000100/2026-14     325
64318 025437/2024-41     277
Name: count, dtype: int64

In [55]:
raw_contratacoes[raw_contratacoes['processo'] == "64318.029552/2024-94"].head(3)

,idcompra,numerocontrolepncp,anocomprapncp,sequencialcomprapncp,orgaoentidadecnpj,orgaosubrogadocnpj,codigoorgao,orgaoentidaderazaosocial,orgaosubrogadorazaosocial,orgaoentidadeesferaid,orgaosubrogadoesferaid,orgaoentidadepoderid,orgaosubrogadopoderid,unidadeorgaocodigounidade,unidadesubrogadacodigounidade,unidadeorgaonomeunidade,unidadesubrogadanomeunidade,unidadeorgaoufsigla,unidadesubrogadaufsigla,unidadeorgaomunicipionome,unidadesubrogadamunicipionome,unidadeorgaocodigoibge,unidadesubrogadacodigoibge,numerocompra,modalidadeidpncp,codigomodalidade,modalidadenome,srp,mododisputaidpncp,codigomododisputa,amparolegalcodigopncp,amparolegalnome,amparolegaldescricao,informacaocomplementar,processo,objetocompra,existeresultado,orcamentosigilosocodigo,orcamentosigilosodescricao,situacaocompraidpncp,situacaocompranomepncp,tipoinstrumentoconvocatoriocodigopncp,tipoinstrumentoconvocatorionome,mododisputanomepncp,valortotalestimado,valortotalhomologado,datainclusaopncp,dataatualizacaopncp,datapublicacaopncp,dataaberturapropostapncp,dataencerramentopropostapncp,contratacaoexcluida,dt_ingest
6746,16055707000752026,00394452000103-1-004172/2026,2026,4172,00394452000103,NaN,44611,COMANDO DO EXERCITO,NaN,F,NaN,E,NaN,160557,NaN,ESCRITÓRIO AVANÇADO DA OP C PIPA 7ª RM,NaN,PE,NaN,PETROLINA,NaN,2611101,NaN,75,9,7,Inexigibilidade,False,5,5,16,"Lei 14.133/2021, Art. 74, IV",Inexigibilidade de Licitação: objetos que devam ou possam ser contratados po...,O Escritório Avançado da Operação Carro-Pipa da 7ª Região Militar (EAOCP/7ª ...,64318.029552/2024-94,"Contratação de serviços de coleta, transporte e distribuição de água potável...",True,1,Compra sem sigilo,1,Divulgada no PNCP,3,Ato que autoriza a Contratação Direta,Não se aplica,151535.2,148080.0,2026-02-26T10:36:46,2026-02-26T10:36:46,2026-02-26T10:36:46,NaN,NaN,False,2026-06-30T17:31:17.225260
7084,16055707000822026,00394452000103-1-004181/2026,2026,4181,00394452000103,NaN,44611,COMANDO DO EXERCITO,NaN,F,NaN,E,NaN,160557,NaN,ESCRITÓRIO AVANÇADO DA OP C PIPA 7ª RM,NaN,PE,NaN,PETROLINA,NaN,2611101,NaN,82,9,7,Inexigibilidade,False,5,5,16,"Lei 14.133/2021, Art. 74, IV",Inexigibilidade de Licitação: objetos que devam ou possam ser contratados po...,NaN,64318.029552/2024-94,"Contratação de serviços de coleta, transporte e distribuição de água potável...",True,1,Compra sem sigilo,1,Divulgada no PNCP,3,Ato que autoriza a Contratação Direta,Não se aplica,149324.8,145920.0,2026-02-26T10:53:37,2026-02-26T10:53:37,2026-02-26T10:53:37,NaN,NaN,False,2026-06-30T17:31:17.225260
7550,16055707001212026,00394452000103-1-004275/2026,2026,4275,00394452000103,NaN,44611,COMANDO DO EXERCITO,NaN,F,NaN,E,NaN,160557,NaN,ESCRITÓRIO AVANÇADO DA OP C PIPA 7ª RM,NaN,PE,NaN,PETROLINA,NaN,2611101,NaN,121,9,7,Inexigibilidade,False,5,5,16,"Lei 14.133/2021, Art. 74, IV",Inexigibilidade de Licitação: objetos que devam ou possam ser contratados po...,O Escritório Avançado da Operação Carro-Pipa da 7ª Região Militar (EAOCP/7ª ...,64318.029552/2024-94,"Contratação de serviços de coleta, transporte e distribuição de água potável...",True,1,Compra sem sigilo,1,Divulgada no PNCP,3,Ato que autoriza a Contratação Direta,Não se aplica,157122.6,153540.0,2026-02-26T15:06:57,2026-02-26T15:06:57,2026-02-26T15:06:57,NaN,NaN,False,2026-06-30T17:31:22.234554


### `objetocompra`
Descrição do objeto da compra

In [56]:
raw_contratacoes.objetocompra.value_counts(0, dropna=False).head()

objetocompra
A presente contratação tem por objeto a contratação de interessados na prestação do serviço de coleta, transporte e distribuição de água potável, para os municípios os estados de Pernambuco e Alagoas que estejam em estado de calamidade pública ou emergencial, escopo da Operação Carro-Pipa.    580
Contratação de serviços de coleta, transporte e distribuição de água potável, para atendimento das necessidades, no particular, de populações situadas em municípios atingidos pela seca, na área do Semiárido Brasileiro.                                                                            562
Prestação dos serviços de coleta, transporte e distribuição de água potável nos municípios em situação declarada de emergência no semiárido nordestino - Operação Carro-Pipa.                                                                                                                         517
Convocação de interessados em se credenciar para prestação de serviços de coleta, transporte 

In [57]:
raw_contratacoes.objetocompra.isna().sum()

0

#### Toda descrição está preenchida, porém sem padronização.

### `existeresultado`
Indicador se há resultado da compra

In [58]:
raw_contratacoes.existeresultado.value_counts(1, dropna=False).head()

existeresultado
True     0.739623
False    0.260377
Name: proportion, dtype: float64

#### 74% das contratações possuem resultado.

### `orcamentosigilosocodigo`
Código do orçamento sigiloso

In [59]:
raw_contratacoes.orcamentosigilosocodigo.value_counts(1, dropna=False).head()

orcamentosigilosocodigo
1    1.0
Name: proportion, dtype: float64

### `orcamentosigilosodescricao`
Descrição do orçamento sigiloso

In [60]:
raw_contratacoes.orcamentosigilosodescricao.value_counts(1, dropna=False).head()

orcamentosigilosodescricao
Compra sem sigilo    1.0
Name: proportion, dtype: float64

#### Apenas compra sem sigilo

### `situacaocompraidpncp`
Identificador da situação da compra no PNCP

In [61]:
raw_contratacoes.situacaocompraidpncp.value_counts(1, dropna=False).head()

situacaocompraidpncp
1    0.974027
4    0.011249
2    0.007900
3    0.006823
Name: proportion, dtype: float64

### `situacaocompranomepncp`
Nome da situação da compra no PNCP

In [62]:
raw_contratacoes.situacaocompranomepncp.value_counts(1, dropna=False).head()

situacaocompranomepncp
Divulgada no PNCP    0.974027
Suspensa             0.011249
Revogada             0.007900
Anulada              0.006823
Name: proportion, dtype: float64

#### Proporção da situação da compra no PNCP
| situacaocompranomepncp   |   Proporção |
|:-------------------------|-------------:|
| Divulgada no PNCP        |         97,4 % |
| Suspensa                 |          1,1 % |
| Revogada                 |          0,8 % |
| Anulada                  |          0,7 % |

### `tipoinstrumentoconvocatoriocodigopncp`
Código do tipo de instrumento convocatório no PNCP

In [63]:
raw_contratacoes.tipoinstrumentoconvocatoriocodigopncp.value_counts(1, dropna=False).head()

tipoinstrumentoconvocatoriocodigopncp
3    0.395539
1    0.391861
2    0.212600
Name: proportion, dtype: float64

### `tipoinstrumentoconvocatorionome`
Nome do tipo de instrumento convocatório

In [64]:
raw_contratacoes.tipoinstrumentoconvocatorionome.value_counts(1, dropna=False).head()

tipoinstrumentoconvocatorionome
Ato que autoriza a Contratação Direta    0.395539
Edital                                   0.391861
Aviso de Contratação Direta              0.212600
Name: proportion, dtype: float64

#### Proporção do tipo de convovação
| tipoinstrumentoconvocatorionome       |   proportion |
|:--------------------------------------|-------------:|
| Ato que autoriza a Contratação Direta |         39.6 |
| Edital                                |         39.2 |
| Aviso de Contratação Direta           |         21.3 |

### `mododisputanomepncp`
Nome do modo de disputa no PNCP

In [65]:
raw_contratacoes.mododisputanomepncp.value_counts(1, dropna=False).head()

mododisputanomepncp
Não se aplica           0.395557
Aberto                  0.241191
Dispensa Com Disputa    0.212600
Aberto-Fechado          0.145254
Fechado                 0.003091
Name: proportion, dtype: float64

#### Proporção do modo de disputa
| mododisputanomepncp   |   proportion |
|:----------------------|-------------:|
| Não se aplica         |         39.6 |
| Aberto                |         24.1 |
| Dispensa Com Disputa  |         21.3 |
| Aberto-Fechado        |         14.5 |
| Fechado               |          0.3 |
| Fechado-Aberto        |          0.2 |

### `valortotalestimado`
Valor total estimado para a compra

In [66]:
raw_contratacoes.valortotalestimado.value_counts(1, dropna=False).head()

valortotalestimado
0.0000         0.057396
174000.0000    0.005879
0.0001         0.004961
161769.3600    0.004329
162000.0000    0.003180
Name: proportion, dtype: float64

#### Quase 6% das compras com valor igual a zero

### `valortotalhomologado`
Valor total homologado para a compra

In [67]:
raw_contratacoes.valortotalhomologado.value_counts(1, dropna=False).head()

valortotalhomologado
NaN            0.260377
0.0001         0.005994
174000.0000    0.005879
161769.3600    0.004338
162000.0000    0.003198
Name: proportion, dtype: float64

#### 26% dos valores nulos

### `datainclusaopncp`
Data de inclusão da compra no PNCP

In [68]:
raw_contratacoes.datainclusaopncp.value_counts(0, dropna=False).head()

datainclusaopncp
2026-05-29T15:31:53    5
2026-04-08T16:00:04    3
2026-04-30T15:44:21    3
2026-05-29T15:27:24    3
2026-02-12T11:09:47    3
Name: count, dtype: int64

### `dataatualizacaopncp`
Data da última atualização da compra no PNCP

In [69]:
raw_contratacoes.dataatualizacaopncp.value_counts(0, dropna=False).head()

dataatualizacaopncp
2026-05-29T15:31:53    5
2026-04-08T16:00:04    3
2026-04-30T15:44:21    3
2026-05-29T15:27:24    3
2026-02-12T11:09:47    3
Name: count, dtype: int64

### `datapublicacaopncp`
Data de publicação da compra no PNCP

In [70]:
raw_contratacoes.datapublicacaopncp.value_counts(0, dropna=False).head()

datapublicacaopncp
2026-05-29T15:31:53    5
2026-04-08T16:00:04    3
2026-04-30T15:44:21    3
2026-05-29T15:27:24    3
2026-02-12T11:09:47    3
Name: count, dtype: int64

### `dataaberturapropostapncp`
Data de abertura das propostas no PNCP

In [71]:
raw_contratacoes.dataaberturapropostapncp.value_counts(1, dropna=False).head()

dataaberturapropostapncp
NaN                    0.395539
2026-05-29T08:00:00    0.006244
2026-05-27T08:00:00    0.006110
2026-05-26T08:00:00    0.005923
2026-05-28T08:00:00    0.005923
Name: proportion, dtype: float64

#### 40% dos valores são nulos

### `dataencerramentopropostapncp`
Data de encerramento das propostas no PNCP

In [72]:
raw_contratacoes.dataencerramentopropostapncp.value_counts(1, dropna=False).head()

dataencerramentopropostapncp
NaN                    0.395539
2026-04-22T08:00:00    0.004436
2026-06-01T08:00:00    0.003358
2026-06-08T08:00:00    0.003331
2026-05-25T08:00:00    0.003304
Name: proportion, dtype: float64

#### 40% dos valores são nulos

### `contratacaoexcluida`
Indica se a contratação foi desconsiderada/excluída

In [73]:
raw_contratacoes.contratacaoexcluida.value_counts(0, dropna=False).head()

contratacaoexcluida
False    112260
True         12
Name: count, dtype: int64

#### Apenas 12 contratações excluídas

## `raw_itens_contratacoes`
Itens individuais de cada processo de contratação.

| Coluna | Descrição |
|---|---|
| `idcompraitem` **PK** | Identificador do item na compra |
| `idcompra` **FK** → `raw_contratacoes` | Compra a que pertence |
| `descricaoresumida` | Descrição resumida do item |
| `descricaodetalhada` | Descrição detalhada |
| `materialouservico` | M (material) ou S (serviço) |
| `coditemcatalogo` | Código no catálogo de materiais/serviços *(join com `raw_item_material.codigoitem` ou `raw_item_servico.codigoservico`)* |
| `codigoclasse` / `codigogrupo` | Classificação no catálogo |
| `quantidade` | Quantidade licitada |
| `valorunitarioestimado` | Valor unitário estimado |
| `situacaocompraitemnome` | Situação do item |
| `temresultado` | Se há resultado homologado |


In [74]:
raw_itens_contratacoes.shape

(243153, 46)

In [75]:
raw_itens_contratacoes.duplicated().sum()

0

In [76]:
raw_itens_contratacoes.head(3)

,idcompra,idcompraitem,idcontratacaopncp,unidadeorgaocodigounidade,orgaoentidadecnpj,numeroitempncp,numeroitemcompra,numerogrupo,descricaoresumida,materialouservico,materialouserviconome,codigoclasse,codigogrupo,coditemcatalogo,descricaodetalhada,unidademedida,orcamentosigiloso,itemcategoriaidpncp,itemcategorianome,criteriojulgamentoidpncp,criteriojulgamentonome,situacaocompraitem,situacaocompraitemnome,tipobeneficio,tipobeneficionome,incentivoprodutivobasico,quantidade,valorunitarioestimado,valortotal,temresultado,codfornecedor,nomefornecedor,quantidaderesultado,valorunitarioresultado,valortotalresultado,datainclusaopncp,dataatualizacaopncp,dataresultado,margempreferencianormal,percentualmargempreferencianormal,margempreferenciaadicional,percentualmargempreferenciaadicional,codigoncm,descricaoncm,numerocontrolepncpcompra,dt_ingest
0,79118007000542026,7911800700054202600001,00394502000144-1-000261/2026,791180,00394502000144,1,1,0,Energia elétrica - fornecimento mercado regulado Energia elétrica - fornecim...,S,Serviço,NaN,691.0,4120.0,Energia elétrica - fornecimento mercado regulado Energia elétrica - fornecim...,UNIDADE,False,3,Informática (TIC),7,Não se aplica,2,Homologado,5,Não se aplica,False,1.0,235403.65,235403.65,True,33050071000158,AMPLA ENERGIA E SERVICOS S.A.,1.0,235403.65,235403.65,2026-01-01T20:02:41,2026-01-01T20:05:43,2026-01-01 00:00:00.0000000,False,NaN,False,NaN,NaN,NaN,00394502000144-1-000261/2026,2026-06-30T13:34:28.795437
1,12063005900362025,1206300590036202500001,00394429000100-1-003992/2025,120630,00394429000100,1,1,0,INSPEÇÕES E TESTES EM MATERIAIS E EQUIPAMENTOS ELÉTRICOS,S,Serviço,NaN,NaN,NaN,NaN,UNIDADE,False,3,Informática (TIC),1,Menor preço,2,Homologado,5,Não se aplica,False,35.0,7612.23,266428.05,True,55751809000147,SMART SOCIEDADE LTDA,35.0,3822.22,133777.70,2026-01-02T07:00:04,2026-02-10T00:01:28,2026-02-10 00:00:00.0000000,False,NaN,False,NaN,NaN,NaN,00394429000100-1-003992/2025,2026-06-30T13:34:28.795437
2,12063005900362025,1206300590036202500002,00394429000100-1-003992/2025,120630,00394429000100,2,2,0,"MANUTENÇÃO / INSTALAÇÃO DE TRANSFORMADORES ACIMA DE 34,5KV",S,Serviço,NaN,NaN,NaN,NaN,UNIDADE,False,3,Informática (TIC),1,Menor preço,2,Homologado,1,Participação exclusiva para ME/EPP,False,3.0,19438.57,58315.71,True,28249240000143,C A RODRIGUES DE ALMEIDA JUNIOR,3.0,12600.00,37800.00,2026-01-02T07:00:04,2026-02-10T00:01:28,2026-02-10 00:00:00.0000000,False,NaN,False,NaN,NaN,NaN,00394429000100-1-003992/2025,2026-06-30T13:34:28.795437


### `idcompra`
Identificador único da compra

In [77]:
raw_itens_contratacoes.idcompra.duplicated().sum()

214591

In [78]:
raw_itens_contratacoes.idcompra.value_counts(0, dropna=False).head()

idcompra
92868005900092026    894
16015505900132025    753
16015905900012026    719
92868005900082026    676
92868005900022026    616
Name: count, dtype: int64

In [79]:
raw_itens_contratacoes[raw_itens_contratacoes['idcompra'] == 92868005900092026].head(2)

,idcompra,idcompraitem,idcontratacaopncp,unidadeorgaocodigounidade,orgaoentidadecnpj,numeroitempncp,numeroitemcompra,numerogrupo,descricaoresumida,materialouservico,materialouserviconome,codigoclasse,codigogrupo,coditemcatalogo,descricaodetalhada,unidademedida,orcamentosigiloso,itemcategoriaidpncp,itemcategorianome,criteriojulgamentoidpncp,criteriojulgamentonome,situacaocompraitem,situacaocompraitemnome,tipobeneficio,tipobeneficionome,incentivoprodutivobasico,quantidade,valorunitarioestimado,valortotal,temresultado,codfornecedor,nomefornecedor,quantidaderesultado,valorunitarioresultado,valortotalresultado,datainclusaopncp,dataatualizacaopncp,dataresultado,margempreferencianormal,percentualmargempreferencianormal,margempreferenciaadicional,percentualmargempreferenciaadicional,codigoncm,descricaoncm,numerocontrolepncpcompra,dt_ingest


#### (TO DO) Investigar os top 10 ids mais repetidos (o primeiro caso parece que adicionaram uma linha pra cada unidade)

### `idcompraitem`
Identificador único do item da compra

In [80]:
raw_itens_contratacoes.idcompraitem.duplicated().sum()

0

#### 243k itens distintos

### `idcontratacaopncp`
Identificador único da contratação
no PNCP

In [81]:
raw_itens_contratacoes.idcontratacaopncp.value_counts(0, dropna=False).head()

idcontratacaopncp
42736102000110-1-000009/2026    894
00394452000103-1-025174/2025    753
00394452000103-1-003861/2026    719
42736102000110-1-000008/2026    676
42736102000110-1-000002/2026    616
Name: count, dtype: int64

### `unidadeorgaocodigounidade`
Código da unidade do órgão

In [82]:
raw_itens_contratacoes.unidadeorgaocodigounidade.value_counts(1, dropna=False).head()

unidadeorgaocodigounidade
928680    0.011441
943001    0.009751
927658    0.009364
925373    0.007567
987541    0.006736
Name: proportion, dtype: float64

### `orgaoentidadecnpj`
CNPJ do órgão ou entidade
responsável pela compra

In [83]:
raw_itens_contratacoes.unidadeorgaocodigounidade.value_counts(1, dropna=False).head()

unidadeorgaocodigounidade
928680    0.011441
943001    0.009751
927658    0.009364
925373    0.007567
987541    0.006736
Name: proportion, dtype: float64

### `numeroitempncp`
Número do item no PNCP

In [84]:
raw_itens_contratacoes.numeroitempncp.value_counts(1, dropna=False).head()

numeroitempncp
1    0.115721
2    0.047320
3    0.036500
4    0.030816
5    0.026720
Name: proportion, dtype: float64

### `numeroitemcompra`
Número do item na compra

In [85]:
raw_itens_contratacoes.numeroitemcompra.value_counts(1, dropna=False).head()

numeroitemcompra
1    0.115721
2    0.047320
3    0.036500
4    0.030816
5    0.026720
Name: proportion, dtype: float64

### `numerogrupo`
Número do grupo ao qual o item
pertence

In [86]:
raw_itens_contratacoes.numerogrupo.value_counts(1, dropna=False).head()

numerogrupo
0    1.0
Name: proportion, dtype: float64

#### (PERGUNTA) Essa coluna não apresenta nenhuma informação (100% das linhas são iguais)

### `descricaoresumida`
Descrição resumida do item

In [87]:
raw_itens_contratacoes.descricaoresumida.value_counts(1, dropna=False).head()

descricaoresumida
Serviço Engenharia                                                                               0.012766
Fruta                                                                                            0.007999
Fornecimento / Transporte de Água - Carro Pipa Fornecimento / Transporte de Água - Carro Pipa    0.006514
Acessórios / Equipamentos Oficina Manutenção tipo: carro esteira, material: aço                  0.005922
Condimento                                                                                       0.005745
Name: proportion, dtype: float64

In [88]:
raw_itens_contratacoes.descricaoresumida.value_counts(0, dropna=False).tail()

descricaoresumida
Compressor De Ar pressão máxima: 115, potência motor: 1,50, vazão: 206, tensão: 110/220, rotação motor: 2.850, capacidade tanque: 0,90, características adicionais: acionamento motor direto, 1 pistão, 2 manômetros Descrição e especificações serão Conforme termo de referência.                                                                                                                         1
Treinamento qualificação profissional cursos promovido por ONE CURSOS - TREINAMENTO E DESENVOLVIMENTO, CNPJ/CPF: 06.012.731/0001-33, a ser realizados online e presenciais, previstos entre março e novembro de 2026.                                                                                                                                                                                       1
Bancada Equipamento Eletrônico material bancada: madeira compensada/mdf e estrutura aço perfurado, material prateleiras: madeira compensada/mdf laminado melamínico cor, material pé: aço,

### `materialouservico`
Indica se é um material ou serviço
(M - Material, S - Serviço)

In [89]:
raw_itens_contratacoes.materialouservico.value_counts(1, dropna=False).head()

materialouservico
M    0.802548
S    0.197452
Name: proportion, dtype: float64

#### 80%/20% Materiais/Serviços

### `materialouserviconome`
Nome do material ou serviço

In [90]:
raw_itens_contratacoes.materialouserviconome.value_counts(1, dropna=False).head()

materialouserviconome
Material    0.802548
Serviço     0.197452
Name: proportion, dtype: float64

#### Dado que são apenas dois valores as duas colunas parecem redundantes.

### `codigoclasse`
Código da classe do item

In [91]:
raw_itens_contratacoes.codigoclasse.value_counts(1, dropna=False).head()

codigoclasse
NaN       0.354596
6515.0    0.072469
6505.0    0.069693
6520.0    0.026773
8915.0    0.023454
Name: proportion, dtype: float64

In [92]:
raw_itens_contratacoes.codigoclasse.value_counts(0, dropna=False).tail()

codigoclasse
5825.0    1
3422.0    1
7435.0    1
3611.0    1
3436.0    1
Name: count, dtype: int64

#### 35% de linhas vazias

### `codigogrupo`
Código do grupo do item

In [93]:
raw_itens_contratacoes.codigogrupo.value_counts(1, dropna=False).head()

codigogrupo
NaN      0.949744
833.0    0.008587
871.0    0.004771
833      0.003652
873.0    0.003471
Name: proportion, dtype: float64

#### 95% de linhas vazias

### `coditemcatalogo`
Código do item no catálogo

In [94]:
raw_itens_contratacoes.coditemcatalogo.value_counts(1, dropna=False).head()

coditemcatalogo
NaN         0.304339
22225.0     0.008131
22225       0.003652
2020.0      0.002570
615119.0    0.001637
Name: proportion, dtype: float64

#### 30% de linhas vazias

### `descricaodetalhada`
Descrição detalhada do item

In [95]:
raw_itens_contratacoes.descricaodetalhada.value_counts(1, dropna=False).head()

descricaodetalhada
NaN                                                                                                        0.304339
LENTE PARA LAMPADA INDICADORA, NOME: LENTE PARA LAMPADA INDICADORA                                         0.011783
EMPILHADEIRA - GARFO, NOME: EMPILHADEIRA - GARFO                                                           0.002570
LIVRO LITERATURA, NÍVEL: ADULTOS , ASSUNTO: LITERATURA , FORMATO: IMPRESSO , IDIOMA: LÍNGUA PORTUGUESA     0.001637
"LEITE PÓ", NOME: LEITE INTEGRAL - PO / LIQUIDO                                                            0.001522
Name: proportion, dtype: float64

In [96]:
raw_itens_contratacoes.descricaodetalhada.value_counts(0, dropna=False).tail()

descricaodetalhada
FLOR NATURAL, ESPÉCIE: ROSA , COR: VARIADA , CARACTERíSTICAS ADICIONAIS: TIPO ARRANJO MÉDIO PARA MESA SOLENE , QUANTIDADE: MIX DE FLORES DIVERSAS                                                                                                                                                1
CABO, MATERIAL: AÇO INOXIDÁVEL , TIPO: ESPELHO DE LARINGE                                                                                                                                                                                                                                        1
FILTRO, NOME: FILTRO DE LUZ                                                                                                                                                                                                                                                                      1
TRATOR DE ESTEIRA, POTÊNCIA MOTOR: 118 HP, PESO: 14.000 KG, TIPO: TURBO ALIMENTADO , SISTEMA TRANSMISSÃO: HI

#### 30% de linhas vazias (TO DO: investigar o motivo ou a fonte)

### `unidademedida`
Unidade de medida do item

In [97]:
raw_itens_contratacoes.unidademedida.value_counts(1, dropna=False)

unidademedida
Unidade                       0.392222
UNIDADE                       0.172760
Unidade                       0.068977
Quilograma                    0.052547
Comprimido                    0.023092
Quilograma                    0.008731
Litro                         0.008229
UNIDADE 0,00                  0.007271
Embalagem 1,00 KG             0.007189
Grama                         0.006338
Frasco-Ampola                 0.005873
Par                           0.005856
Embalagem 500,00 G            0.005704
METRO CÚBICO                  0.005420
Metro                         0.004890
Teste                         0.004857
Comprimido                    0.004524
Caixa 100,00 UN               0.003969
Pacote 100,00 UN              0.003932
METRO QUADRADO                0.003730
Frasco 100,00 ML              0.003627
Ampola 2,00 ML                0.003463
Ampola 1,00 ML                0.003438
Frasco 10,00 ML               0.003130
Embalagem 100,00 UN           0.002941
Cápsula    

#### Inclui a unidade de medida e por vezes o volume ou peso, container, descrição.
Falta padronização, recomenda-se limitar o input. Também pode auxiliar diagnóstico, pode representar informações relevantes que não são possíveis de inserir em outro campo.

### `orcamentosigiloso`
Indicador se o orçamento é
sigiloso 

0 – False/Não

1 – True/Sim

In [98]:
raw_itens_contratacoes.orcamentosigiloso.value_counts(1, dropna=False).head()

orcamentosigiloso
False    0.813874
True     0.186126
Name: proportion, dtype: float64

#### 18% se referem ao orçamento sigiloso

### `itemcategoriaidpncp`
Identificador da categoria do item
no PNCP

In [99]:
raw_itens_contratacoes.itemcategoriaidpncp.value_counts(1, dropna=False).head()

itemcategoriaidpncp
3    1.0
Name: proportion, dtype: float64

### `itemcategorianome`
Nome da categoria do item

In [100]:
raw_itens_contratacoes.itemcategorianome.value_counts(1, dropna=False).head()

itemcategorianome
Informática (TIC)    1.0
Name: proportion, dtype: float64

#### * **100% do itens estão na categoria Informática**

### `criteriojulgamentoidpncp`
Identificador do critério de
julgamento no PNCP

In [101]:
raw_itens_contratacoes.criteriojulgamentoidpncp.value_counts(1, dropna=False).head()

criteriojulgamentoidpncp
1    0.874071
7    0.113270
2    0.011384
4    0.001127
6    0.000099
Name: proportion, dtype: float64

### `criteriojulgamentonome`
Nome do critério de julgamento

In [102]:
raw_itens_contratacoes.criteriojulgamentonome.value_counts(1, dropna=False)

criteriojulgamentonome
Menor preço                0.874071
Não se aplica              0.113270
Maior desconto             0.011384
Técnica e preço            0.001127
Maior retorno econômico    0.000099
Melhor técnica             0.000025
Conteúdo artístico         0.000025
Name: proportion, dtype: float64

#### 87% de Menor Preço
Caveat: Os itens que foram inseridos em mais linhas vão ter um peso maior.

### `situacaocompraitem`
Situação do item da compra

In [103]:
raw_itens_contratacoes.situacaocompraitem.value_counts(1, dropna=False).head()

situacaocompraitem
2    0.689266
1    0.176909
5    0.078025
3    0.033991
4    0.021809
Name: proportion, dtype: float64

### `situacaocompraitemnome`
Nome da situação do item da
compra

In [104]:
raw_itens_contratacoes.situacaocompraitemnome.value_counts(1, dropna=False)

situacaocompraitemnome
Homologado                    0.689266
Em andamento                  0.176909
Fracassado                    0.078025
Anulado/Revogado/Cancelado    0.033991
Deserto                       0.021809
Name: proportion, dtype: float64

#### 17% em andamento, 2% Deserto

### `tipobeneficio`
Tipo de benefício associado ao
item

In [105]:
raw_itens_contratacoes.tipobeneficio.value_counts(1, dropna=False)

tipobeneficio
5    0.507142
1    0.486249
3    0.006510
2    0.000053
4    0.000045
Name: proportion, dtype: float64

### `tipobeneficionome`
Nome do tipo de benefício
associado

In [106]:
raw_itens_contratacoes.tipobeneficionome.value_counts(1, dropna=False)

tipobeneficionome
Não se aplica                         0.507142
Participação exclusiva para ME/EPP    0.486249
Cota reservada para ME/EPP            0.006510
Subcontratação para ME/EPP            0.000053
Sem benefício                         0.000045
Name: proportion, dtype: float64

#### 50% de `Não se aplica` e 0,0045% de `Sem benefício`

#### 48% de Participação exclusiva para ME/EPP

### `incentivoprodutivobasico`
Indicador de incentivo produtivo
básico
0 – False/Não
1 – True/Sim

In [107]:
raw_itens_contratacoes.incentivoprodutivobasico.value_counts(1, dropna=False).head()

incentivoprodutivobasico
False    1.0
Name: proportion, dtype: float64

#### * 100% de Falsos

### `quantidade`
Quantidade do item

In [108]:
raw_itens_contratacoes.quantidade.value_counts(0, dropna=False).head()

quantidade
1.0      24903
10.0     10261
2.0       9887
20.0      8110
100.0     7972
Name: count, dtype: int64

In [109]:
raw_itens_contratacoes.quantidade.value_counts(0, dropna=False).tail()

quantidade
1750000.0    1
1737611.0    1
304505.0     1
600013.0     1
305096.0     1
Name: count, dtype: int64

### `valorunitarioestimado`
Valor unitário estimado do item

In [110]:
raw_itens_contratacoes.valorunitarioestimado.value_counts(1, dropna=False).head()

valorunitarioestimado
0.00        0.092814
1.00        0.009241
270.00      0.002743
70.58       0.001888
29000.00    0.001785
Name: proportion, dtype: float64

In [111]:
raw_itens_contratacoes.valorunitarioestimado.value_counts(0, dropna=False).tail()

valorunitarioestimado
2012.10     1
58290.00    1
12970.52    1
7176.00     1
55450.00    1
Name: count, dtype: int64

#### 9% estimado a 0

### `valortotal`
Valor total estimado do item

In [112]:
raw_itens_contratacoes.valorunitarioestimado.value_counts(1, dropna=False).head()

valorunitarioestimado
0.00        0.092814
1.00        0.009241
270.00      0.002743
70.58       0.001888
29000.00    0.001785
Name: proportion, dtype: float64

### `temresultado`
Indicador se há resultado para o
item

0 – False/Não

1 – True/Sim

In [113]:
raw_itens_contratacoes.temresultado.value_counts(1, dropna=False).head()

temresultado
True    0.693066
NaN     0.306934
Name: proportion, dtype: float64

#### 30% de valores vazios, os outros 70% são todos verdadeiros.
Os valores vazios talvez sejam correspondentes aos Falsos.

### `codfornecedor`
Código do fornecedor do item

In [114]:
raw_itens_contratacoes.codfornecedor.value_counts(1, dropna=False).head()

codfornecedor
NaN               0.305260
46884097000143    0.003825
28820255000110    0.003668
53804203000105    0.002093
12433700000159    0.001970
Name: proportion, dtype: float64

In [115]:
raw_itens_contratacoes.codfornecedor.value_counts(0, dropna=False).tail()

codfornecedor
05428673000160    1
02261854000157    1
07080227000133    1
04345304000141    1
10976851000128    1
Name: count, dtype: int64

#### 30% dos valores vazios
#### (TO DO): Investigar o motivo/fonte de 30% dos valores vazios de fornecedor

### `nomefornecedor`
Nome do fornecedor do item


In [116]:
raw_itens_contratacoes.nomefornecedor.value_counts(1, dropna=False).head()

nomefornecedor
NaN                                                                                0.305260
GOLDEN PRODUTOS ODONTOLOGICOS LTDA                                                 0.003825
SUPREMA DENTAL IMPORTACAO, EXPORTACAO E COMERCIO DE PRODUTOS ODONTOLOGICOS LTDA    0.003668
KLAS SUPRIMENTOS LTDA                                                              0.002093
NUTRICELLI COMERCIO DE ALIMENTOS LTDA                                              0.001970
Name: proportion, dtype: float64

In [117]:
raw_itens_contratacoes.nomefornecedor.value_counts(0, dropna=False).tail()

nomefornecedor
SIGN CUT COMUNICACAO VISUAL LTDA     1
UNINTER EDUCACIONAL S/A              1
PORTES CLINICA E ODONTOLOGIA LTDA    1
ELLAN S/A                            1
BRAIN BASSI & NASCIMENTO LTDA        1
Name: count, dtype: int64

#### Top 10 Fornecedores na base
| nomefornecedor                                                                  |   proportion |
|:--------------------------------------------------------------------------------|-------------:|
| nan                                                                             |        30.53 |
| GOLDEN PRODUTOS ODONTOLOGICOS LTDA                                              |         0.38 |
| SUPREMA DENTAL IMPORTACAO, EXPORTACAO E COMERCIO DE PRODUTOS ODONTOLOGICOS LTDA |         0.37 |
| KLAS SUPRIMENTOS LTDA                                                           |         0.21 |
| NUTRICELLI COMERCIO DE ALIMENTOS LTDA                                           |         0.2  |
| SOUL DISTRIBUIDORA DE PRODUTOS E EQUIPAMENTOS INDUSTRIAIS LTDA                  |         0.19 |
| GENTE SEGURADORA SA                                                             |         0.19 |
| AZ&HE PLAN ENGENHARIA S/A                                                       |         0.18 |
| E.C.S. TECNOLOGIA EM SAUDE, COMERCIO, IMPORTACAO E EXPORTACAO LTDA              |         0.18 |
| DPEL DISTRIBUIDORA DE PAPELARIA E LIMPEZA LTDA                                  |         0.18 |
| CREATIVE THINGS COMERCIO DE DISTRIBUICAO LTDA                                   |         0.18 |

### `quantidaderesultado`
Quantidade do item no resultado

In [118]:
raw_itens_contratacoes.quantidaderesultado.value_counts(1, dropna=False).head()

quantidaderesultado
NaN     0.305260
1.0     0.079279
10.0    0.028723
2.0     0.028213
20.0    0.022772
Name: proportion, dtype: float64

### `valorunitarioresultado`
Valor unitário do item no resultado

In [119]:
raw_itens_contratacoes.valorunitarioresultado.value_counts(1, dropna=False).head()

valorunitarioresultado
NaN     0.305260
1.0     0.007687
15.0    0.002533
5.0     0.002484
10.0    0.002394
Name: proportion, dtype: float64

### `valortotalresultado`
Valor total do item no resultado

In [120]:
raw_itens_contratacoes.valortotalresultado.value_counts(1, dropna=False).head()

valortotalresultado
NaN          0.305260
174000.00    0.002410
161769.36    0.001855
6000.00      0.001805
600.00       0.001612
Name: proportion, dtype: float64

In [121]:
raw_itens_contratacoes[raw_itens_contratacoes['valortotal'] == raw_itens_contratacoes['valortotalresultado']].shape[0]/raw_itens_contratacoes[raw_itens_contratacoes['valortotalresultado'].notnull()].shape[0]

0.19803111384731956

In [122]:
raw_itens_contratacoes[raw_itens_contratacoes['valortotal'] > raw_itens_contratacoes['valortotalresultado']].shape[0]/raw_itens_contratacoes[raw_itens_contratacoes['valortotalresultado'].notnull()].shape[0]

0.7946935972722107

In [123]:
raw_itens_contratacoes[raw_itens_contratacoes['valortotal'] < raw_itens_contratacoes['valortotalresultado']].shape[0]/raw_itens_contratacoes[raw_itens_contratacoes['valortotalresultado'].notnull()].shape[0]

0.007275288880469786

#### 20% tem resultado igual ao estimado
#### 79% tem resultado menor que o estimado
#### 0,7% tem resultado maior que o estimado

### `datainclusaopncp`
Data de inclusão do item no PNCP

In [124]:
raw_itens_contratacoes.datainclusaopncp.value_counts(0, dropna=False).head()

datainclusaopncp
2026-02-10T07:02:28    894
2026-02-04T07:02:31    753
2026-02-23T07:10:34    719
2026-02-05T07:05:52    676
2026-01-22T07:26:58    616
Name: count, dtype: int64

#### (TO DO) Investigar linhas inseridar no mesmo segundo (talvez a fonte)

### `dataatualizacaopncp`
Data da última atualização do item
no PNCP

In [125]:
raw_itens_contratacoes.dataatualizacaopncp.value_counts(1, dropna=False).head()

dataatualizacaopncp
2026-02-04T07:02:31    0.002336
2026-01-22T07:01:26    0.001925
2026-01-26T07:04:40    0.001921
2026-02-05T07:14:04    0.001838
2026-02-20T07:00:53    0.001546
Name: proportion, dtype: float64

### `dataresultado`
Data do resultado para o item

In [126]:
raw_itens_contratacoes.dataresultado.value_counts(1, dropna=False).head()

dataresultado
NaN                            0.305260
2026-03-06 00:00:00.0000000    0.018577
2026-03-05 00:00:00.0000000    0.016566
2026-03-10 00:00:00.0000000    0.016430
2026-03-04 00:00:00.0000000    0.015056
Name: proportion, dtype: float64

In [127]:
raw_itens_contratacoes.dataresultado.value_counts(0, dropna=False).head()

dataresultado
NaN                            74225
2026-03-06 00:00:00.0000000     4517
2026-03-05 00:00:00.0000000     4028
2026-03-10 00:00:00.0000000     3995
2026-03-04 00:00:00.0000000     3661
Name: count, dtype: int64

Ter data do resultado igual é mais comum, já que é medido por dia.

### `margempreferencianormal`
Indicador de margem de
preferência normal

0 – False/Não

1 – True/Sim

In [128]:
raw_itens_contratacoes.margempreferencianormal.value_counts(1, dropna=False).head()

margempreferencianormal
False    0.98934
True     0.01066
Name: proportion, dtype: float64

### 1% tem margem de preferencia normal

### `percentualmargempreferencianormal`
Percentual da margem de preferência normal

In [129]:
raw_itens_contratacoes.percentualmargempreferencianormal.value_counts(1, dropna=True)

percentualmargempreferencianormal
10.0    0.758873
5.0     0.241127
Name: proportion, dtype: float64

#### 75% tem 10 de margem e 25% tem 5 de margem

### `margempreferenciaadicional`
Indicador de margem de
preferência adicional

0 – False/Não

1 – True/Sim

In [130]:
raw_itens_contratacoes.margempreferenciaadicional.value_counts(1, dropna=False)

margempreferenciaadicional
False    0.996332
True     0.003668
Name: proportion, dtype: float64

#### 0,3% tem margem adicional

In [131]:
raw_itens_contratacoes[(raw_itens_contratacoes['margempreferenciaadicional'] == False) & (raw_itens_contratacoes['margempreferencianormal'] == True)].shape[0]/raw_itens_contratacoes[(raw_itens_contratacoes['margempreferenciaadicional'] == True) | (raw_itens_contratacoes['margempreferencianormal'] == True)].shape[0]

0.6558641975308642

In [132]:
raw_itens_contratacoes[(raw_itens_contratacoes['margempreferenciaadicional'] == True) & (raw_itens_contratacoes['margempreferencianormal'] == True)].shape[0]/raw_itens_contratacoes[(raw_itens_contratacoes['margempreferenciaadicional'] == True) | (raw_itens_contratacoes['margempreferencianormal'] == True)].shape[0]

0.3441358024691358

In [133]:
raw_itens_contratacoes[(raw_itens_contratacoes['margempreferenciaadicional'] == False) & (raw_itens_contratacoes['margempreferencianormal'] == False)].shape[0]/raw_itens_contratacoes[(raw_itens_contratacoes['margempreferenciaadicional'] == True) | (raw_itens_contratacoes['margempreferencianormal'] == True)].shape[0]

92.80902777777777

In [134]:
raw_itens_contratacoes[(raw_itens_contratacoes['margempreferenciaadicional'] == True) & (raw_itens_contratacoes['margempreferencianormal'] == False)].shape

(0, 46)

In [135]:
raw_itens_contratacoes[(raw_itens_contratacoes['margempreferenciaadicional'] == True) | (raw_itens_contratacoes['margempreferencianormal'] == True)].shape[0]

2592

#### (TO DO) RETORNAR AQUI

### `percentualmargempreferenciaadicional`
Percentual da margem de preferência adicional

In [136]:
raw_itens_contratacoes.percentualmargempreferenciaadicional.value_counts(1, dropna=True).head()

percentualmargempreferenciaadicional
10.00    0.994395
1.00     0.004484
0.01     0.001121
Name: proportion, dtype: float64

#### 99% tem 10 de margem adicional, 0,4% tem 1 de margem adiconal e 0,1% tem 0.01 de margem adicional

### `codigoncm`
Código da Nomenclatura Comum do Mercosul (NCM)

In [137]:
raw_itens_contratacoes.codigoncm.value_counts(0, dropna=False).head()

codigoncm
NaN         210271
30049069      3265
30049099      2450
30061090      1557
30049079      1357
Name: count, dtype: int64

In [138]:
raw_itens_contratacoes.codigoncm.value_counts(0, dropna=False).tail()

codigoncm
84716090    1
84433222    1
30021510    1
84818099    1
85235290    1
Name: count, dtype: int64

### `descricaoncm`
Descrição da Nomenclatura Comum do Mercosul (NCM)

In [139]:
raw_itens_contratacoes.descricaoncm.value_counts(1, dropna=False).head()

descricaoncm
NaN    1.0
Name: proportion, dtype: float64

#### Coluna vazia

### `numerocontrolepncpcompra`
#### Não parece estar na documentação da API e parece ser igual à coluna `numerocontrolepncp`

In [140]:
raw_itens_contratacoes.numerocontrolepncpcompra.value_counts(0, dropna=True).head()

numerocontrolepncpcompra
42736102000110-1-000009/2026    894
00394452000103-1-025174/2025    753
00394452000103-1-003861/2026    719
42736102000110-1-000008/2026    676
42736102000110-1-000002/2026    616
Name: count, dtype: int64

In [141]:
raw_itens_contratacoes.numerocontrolepncpcompra.value_counts(0, dropna=True).tail()

numerocontrolepncpcompra
00038166000105-1-000107/2026    1
46068425000133-1-000370/2026    1
46068425000133-1-000371/2026    1
78348059000162-1-000012/2026    1
00414607000118-1-000044/2026    1
Name: count, dtype: int64

## `raw_resultado_itens_contratacoes`
Resultado de cada item — fornecedor vencedor e valor homologado.

| Coluna | Descrição |
|---|---|
| `idcompraitem` **PK/FK** → `raw_itens_contratacoes` | Item da contratação |
| `sequencialresultado` **PK** | Sequência do resultado (pode haver mais de um por item em SRP) |
| `nifornecedor` | CPF/CNPJ do fornecedor vencedor |
| `nomerazaosocialfornecedor` | Nome do fornecedor |
| `quantidadehomologada` | Quantidade efetivamente homologada |
| `valorunitariohomologado` | Valor unitário final |
| `valortotalhomologado` | Valor total homologado |
| `portefornecedornome` | Porte do fornecedor (ME, EPP, Grande Empresa) |
| `situacaocompraitemresultadonome` | Situação do resultado |


In [142]:
raw_resultado_itens_contratacoes.shape

(332987, 43)

In [143]:
raw_resultado_itens_contratacoes.duplicated().sum()

0

In [144]:
raw_resultado_itens_contratacoes.head()

,idcompraitem,idcompra,idcontratacaopncp,unidadeorgaocodigounidade,unidadeorgaoufsigla,numeroitempncp,sequencialresultado,nifornecedor,tipopessoa,nomerazaosocialfornecedor,codigopais,indicadorsubcontratacao,ordemclassificacaosrp,quantidadehomologada,valorunitariohomologado,valortotalhomologado,percentualdesconto,situacaocompraitemresultadoid,situacaocompraitemresultadonome,motivocancelamento,portefornecedorid,portefornecedornome,naturezajuridicanome,naturezajuridicaid,datainclusaopncp,dataatualizacaopncp,datacancelamentopncp,dataresultadopncp,numerocontrolepncpcompra,orgaoentidadecnpj,aplicacaomargempreferencia,amparolegalmargempreferenciaid,amparolegalmargempreferencianome,aplicacaobeneficiomeepp,aplicacaocriteriodesempate,amparolegalcriteriodesempateid,amparolegalcriteriodesempatenome,moedaestrangeiraid,datacotacaomoedaestrangeira,valornominalmoedaestrangeira,paisorigemprodutoservicoid,timezonecotacaomoedaestrangeira,dt_ingest
0,9900350590003202500002,99003505900032025,08755269000190-1-000028/2025,990035,SP,2,1,28673951000140,PJ,TECKMAX COMERCIO DE MOVEIS LTDA,BRA,False,NaN,20.0,510.0,10200.0,0.0,1,Informado,NaN,1,ME,Sociedade Empresária Limitada,2062,2026-01-02T14:16:02,2026-01-02T14:16:02,NaN,2026-01-02T00:00:00,08755269000190-1-000028/2025,08755269000190,False,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30T13:34:28.818945
1,9900350590003202500003,99003505900032025,08755269000190-1-000028/2025,990035,SP,3,1,24487206000156,PJ,SEAOPEN REFRIGERACAO E MOVEIS LTDA,BRA,False,NaN,40.0,905.0,36200.0,0.0,1,Informado,NaN,1,ME,Sociedade Empresária Limitada,2062,2026-01-02T14:16:02,2026-01-02T14:16:02,NaN,2026-01-02T00:00:00,08755269000190-1-000028/2025,08755269000190,False,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30T13:34:28.818945
2,9900350590003202500005,99003505900032025,08755269000190-1-000028/2025,990035,SP,5,1,18493830000163,PJ,META X INDUSTRIA E COMERCIO LTDA,BRA,False,NaN,106.0,900.0,95400.0,0.0,1,Informado,NaN,3,Demais,Sociedade Empresária Limitada,2062,2026-01-02T14:16:02,2026-01-02T14:16:02,NaN,2026-01-02T00:00:00,08755269000190-1-000028/2025,08755269000190,False,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30T13:34:28.818945
3,9900350590003202500001,99003505900032025,08755269000190-1-000028/2025,990035,SP,1,1,28673951000140,PJ,TECKMAX COMERCIO DE MOVEIS LTDA,BRA,False,NaN,20.0,710.0,14200.0,0.0,1,Informado,NaN,1,ME,Sociedade Empresária Limitada,2062,2026-01-02T14:16:02,2026-01-02T14:16:02,NaN,2026-01-02T00:00:00,08755269000190-1-000028/2025,08755269000190,False,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30T13:34:28.818945
4,9250900600006202500008,92509006000062025,05535758000148-1-000031/2025,925090,SP,8,1,21205134000119,PJ,CLS EXTINTORES E ENGENHARIA DE COMBATE A INCENDIO LTDA,BRA,False,NaN,1.0,69.0,69.0,0.0,1,Informado,NaN,1,ME,Sociedade Empresária Limitada,2062,2026-01-02T11:54:56,2026-01-02T11:54:56,NaN,2026-01-02T00:00:00,05535758000148-1-000031/2025,05535758000148,False,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30T13:34:28.818945


### `idcompraitem`
Identificador único do item da compra

In [145]:
raw_resultado_itens_contratacoes.idcompraitem.value_counts(0, dropna=False).head()

idcompraitem
1605000000002202600001    428
1604930000035202600001    350
1604920000287202600001    313
1605520000115202500001    256
1605570000030202500001    253
Name: count, dtype: int64

In [146]:
raw_resultado_itens_contratacoes.idcompraitem.value_counts(0, dropna=False).tail()

idcompraitem
1559140590060202500097    1
1559140590060202500095    1
1559140590060202500098    1
1559140590060202500099    1
1559140590060202500101    1
Name: count, dtype: int64

### `idcompra`
Identificador único da compra

In [147]:
raw_resultado_itens_contratacoes.idcompra.value_counts(0, dropna=False).head()

idcompra
12064505900172025    2061
92765800000012026    1873
98769305900612025    1341
18302305900202025    1184
15316505900122025     682
Name: count, dtype: int64

In [148]:
raw_resultado_itens_contratacoes.idcompra.value_counts(0, dropna=False).tail()

idcompra
15590107000132026    1
15501106000262026    1
15591307000622026    1
15590307000112026    1
15500807000042026    1
Name: count, dtype: int64

### `idcontratacaopncp`
Identificador único da contratação no PNCP

In [149]:
raw_resultado_itens_contratacoes.idcontratacaopncp.value_counts(0, dropna=False).head()

idcontratacaopncp
00394429000100-1-000524/2025    2061
02334933000140-1-000001/2026    1873
76995323000124-1-000245/2025    1341
00662270000168-1-000164/2025    1184
24416174000106-1-000126/2025     682
Name: count, dtype: int64

In [150]:
raw_resultado_itens_contratacoes.idcontratacaopncp.value_counts(0, dropna=False).tail()

idcontratacaopncp
15126437000305-1-000921/2026    1
15126437000305-1-000923/2026    1
15126437000305-1-000925/2026    1
15126437000305-1-000949/2026    1
15126437000305-1-001004/2026    1
Name: count, dtype: int64

### `unidadeorgaocodigounidade`
Código da unidade do órgão

In [151]:
raw_resultado_itens_contratacoes.unidadeorgaocodigounidade.value_counts(0, dropna=False).head()

unidadeorgaocodigounidade
927658    4367
943001    3187
120645    2123
927996    1943
926150    1821
Name: count, dtype: int64

In [152]:
raw_resultado_itens_contratacoes.unidadeorgaocodigounidade.value_counts(0, dropna=False).tail()

unidadeorgaocodigounidade
170346    1
373032    1
158289    1
158478    1
158364    1
Name: count, dtype: int64

### `unidadeorgaoufsigla`
Sigla da unidade federativa do órgão

In [153]:
raw_resultado_itens_contratacoes.unidadeorgaoufsigla.value_counts(1, dropna=False).head()

unidadeorgaoufsigla
SP    0.162376
PR    0.124374
RJ    0.105037
MG    0.075910
TO    0.053936
Name: proportion, dtype: float64

In [154]:
raw_resultado_itens_contratacoes.unidadeorgaoufsigla.value_counts(1, dropna=False).tail()

unidadeorgaoufsigla
RR     0.011322
MT     0.010223
AP     0.006610
SE     0.003264
NaN    0.000003
Name: proportion, dtype: float64

### `numeroitempncp`
Número do item divulgado no PNCP

In [155]:
raw_resultado_itens_contratacoes.numeroitempncp.value_counts(0, dropna=False).head()

numeroitempncp
1    42372
2    15517
3    12237
4    10052
5     8772
Name: count, dtype: int64

In [156]:
raw_resultado_itens_contratacoes.numeroitempncp.value_counts(0, dropna=False).tail()

numeroitempncp
2419    1
2340    1
2352    1
2406    1
2413    1
Name: count, dtype: int64

### `sequencialresultado`
Número sequencial do resultado

In [157]:
raw_resultado_itens_contratacoes.sequencialresultado.value_counts(0, dropna=False).head()

sequencialresultado
1    320309
2      5080
3       805
4       491
5       387
Name: count, dtype: int64

In [158]:
raw_resultado_itens_contratacoes.sequencialresultado.value_counts(0, dropna=False).tail()

sequencialresultado
1453    1
1454    1
476     1
477     1
478     1
Name: count, dtype: int64

### `nifornecedor`
Identificador nacional do fornecedor

In [159]:
raw_resultado_itens_contratacoes.nifornecedor.value_counts(0, dropna=False).head()

nifornecedor
46884097000143    1862
51659136000149    1354
04325195000109    1177
54303736000168     986
44734671002286     971
Name: count, dtype: int64

In [160]:
raw_resultado_itens_contratacoes.nifornecedor.value_counts(0, dropna=False).tail()

nifornecedor
23559275000165    1
27550893000367    1
22802705000165    1
04324939000162    1
29412918000120    1
Name: count, dtype: int64

### `tipopessoa`
Tipo de pessoa do fornecedor (física ou jurídica)

In [161]:
raw_resultado_itens_contratacoes.tipopessoa.value_counts(1, dropna=False)

tipopessoa
PJ    0.968182
PF    0.029932
PE    0.001886
Name: proportion, dtype: float64

In [162]:
raw_resultado_itens_contratacoes.tipopessoa.value_counts(0, dropna=False)

tipopessoa
PJ    322392
PF      9967
PE       628
Name: count, dtype: int64

In [163]:
raw_resultado_itens_contratacoes[raw_resultado_itens_contratacoes['tipopessoa'] == "PE"].head()

,idcompraitem,idcompra,idcontratacaopncp,unidadeorgaocodigounidade,unidadeorgaoufsigla,numeroitempncp,sequencialresultado,nifornecedor,tipopessoa,nomerazaosocialfornecedor,codigopais,indicadorsubcontratacao,ordemclassificacaosrp,quantidadehomologada,valorunitariohomologado,valortotalhomologado,percentualdesconto,situacaocompraitemresultadoid,situacaocompraitemresultadonome,motivocancelamento,portefornecedorid,portefornecedornome,naturezajuridicanome,naturezajuridicaid,datainclusaopncp,dataatualizacaopncp,datacancelamentopncp,dataresultadopncp,numerocontrolepncpcompra,orgaoentidadecnpj,aplicacaomargempreferencia,amparolegalmargempreferenciaid,amparolegalmargempreferencianome,aplicacaobeneficiomeepp,aplicacaocriteriodesempate,amparolegalcriteriodesempateid,amparolegalcriteriodesempatenome,moedaestrangeiraid,datacotacaomoedaestrangeira,valornominalmoedaestrangeira,paisorigemprodutoservicoid,timezonecotacaomoedaestrangeira,dt_ingest
34,0100010700012202600001,01000107000122026,00530352000159-1-000007/2026,010001,DF,1,1,EX8174361,PE,INFORMA UK LIMITED,GBR,False,1.0,1.0,10125.07,10125.07,0.0,1,Informado,NaN,5,Não Informado,NaN,NaN,2026-01-02T15:26:08,2026-01-02T15:26:08,NaN,2026-01-02T00:00:00,00530352000159-1-000007/2026,00530352000159,False,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30T13:34:28.818945
5407,0923010590729202500001,09230105907292025,60448040000122-1-000716/2025,092301,SP,1,1,ESTRANG0002597,PE,Elekta Solutions AB,SWE,False,NaN,1.0,11500000.00,11500000.00,0.0,1,Informado,NaN,4,Não se aplica,NaN,NaN,2026-01-05T16:06:45,2026-01-05T16:06:45,NaN,2026-01-05T00:00:00,60448040000122-1-000716/2025,60448040000122,False,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30T13:34:55.094288
5408,0923010590729202500002,09230105907292025,60448040000122-1-000716/2025,092301,SP,2,1,ESTRANG0002597,PE,Elekta Solutions AB,SWE,False,NaN,1.0,10200000.00,10200000.00,0.0,1,Informado,NaN,4,Não se aplica,NaN,NaN,2026-01-05T16:08:45,2026-01-05T16:08:45,NaN,2026-01-05T00:00:00,60448040000122-1-000716/2025,60448040000122,False,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30T13:34:55.094288
8521,1132140700040202600001,11321407000402026,07947821000189-1-000001/2026,113214,DF,1,1,EX0000000,PE,SAE INTERNATIONAL,USA,False,1.0,1.0,1375.40,1375.40,0.0,1,Informado,NaN,5,Não Informado,NaN,NaN,2026-01-06T08:34:28,2026-01-06T08:34:28,NaN,2026-01-06T00:00:00,07947821000189-1-000001/2026,07947821000189,False,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30T13:35:16.622790
10153,3900070700005202600001,39000707000052026,37115342000167-1-000002/2026,390007,DF,1,1,530196932,PE,National Academies of Engineering,USA,False,1.0,1.0,26372.94,26372.94,0.0,1,Informado,NaN,5,Não Informado,NaN,NaN,2026-01-06T10:23:20,2026-01-06T10:23:20,NaN,2026-01-06T00:00:00,37115342000167-1-000002/2026,37115342000167,False,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30T13:35:23.721219


In [164]:
raw_resultado_itens_contratacoes[raw_resultado_itens_contratacoes['tipopessoa'] == "PE"].tail()

,idcompraitem,idcompra,idcontratacaopncp,unidadeorgaocodigounidade,unidadeorgaoufsigla,numeroitempncp,sequencialresultado,nifornecedor,tipopessoa,nomerazaosocialfornecedor,codigopais,indicadorsubcontratacao,ordemclassificacaosrp,quantidadehomologada,valorunitariohomologado,valortotalhomologado,percentualdesconto,situacaocompraitemresultadoid,situacaocompraitemresultadonome,motivocancelamento,portefornecedorid,portefornecedornome,naturezajuridicanome,naturezajuridicaid,datainclusaopncp,dataatualizacaopncp,datacancelamentopncp,dataresultadopncp,numerocontrolepncpcompra,orgaoentidadecnpj,aplicacaomargempreferencia,amparolegalmargempreferenciaid,amparolegalmargempreferencianome,aplicacaobeneficiomeepp,aplicacaocriteriodesempate,amparolegalcriteriodesempateid,amparolegalcriteriodesempatenome,moedaestrangeiraid,datacotacaomoedaestrangeira,valornominalmoedaestrangeira,paisorigemprodutoservicoid,timezonecotacaomoedaestrangeira,dt_ingest
326233,1021030700747202600001,10210307007472026,63025530000104-1-000712/2026,102103,SP,1,1,000000000,PE,Doshin EC Inc,JPN,False,1.0,1.0,3083.63,3083.63,0.0,1,Informado,NaN,5,Não Informado,NaN,NaN,2026-03-16T14:05:06,2026-03-16T14:05:06,NaN,2026-03-16T00:00:00,63025530000104-1-000712/2026,63025530000104,False,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30T14:36:28.711925
326372,9282780390065202500003,92827803900652025,68596162000178-1-000324/2025,928278,PR,3,1,ESTRANG0002712,PE,GAMIC GmbH,DEU,False,NaN,1.0,5839816.66,5839816.66,0.0,1,Informado,NaN,4,Não se aplica,NaN,NaN,2026-03-16T16:08:31,2026-03-16T16:08:31,NaN,2026-03-16T00:00:00,68596162000178-1-000324/2025,68596162000178,False,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30T14:36:34.214938
327851,1790870700368202600001,17908707003682026,00038166000105-1-000176/2026,179087,DF,1,1,EX585UNLO,PE,Université Catholique de Louvain,BEL,False,1.0,2.0,7700.00,15400.00,0.0,1,Informado,NaN,5,Não Informado,NaN,NaN,2026-03-17T11:16:25,2026-03-17T11:16:25,NaN,2026-03-17T00:00:00,00038166000105-1-000176/2026,00038166000105,False,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30T14:37:03.910271
330104,2401260700039202600001,24012607000392026,01263896000164-1-000146/2026,240126,RJ,1,1,GB21-740-8160,PE,IOP PUBLISHING LIMITED,GBR,False,1.0,1.0,700.00,700.00,0.0,1,Informado,NaN,5,Não Informado,NaN,NaN,2026-03-17T08:16:10,2026-03-17T08:16:10,NaN,2026-03-17T00:00:00,01263896000164-1-000146/2026,01263896000164,False,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30T14:37:29.662406
331014,1132140700068202600001,11321407000682026,07947821000189-1-000022/2026,113214,DF,1,1,EX0000000,PE,UNIVERSITY OF KANSAS (KU),USA,False,1.0,1.0,14421.10,14421.10,0.0,1,Informado,NaN,5,Não Informado,NaN,NaN,2026-03-17T13:41:59,2026-03-17T13:41:59,NaN,2026-03-17T00:00:00,07947821000189-1-000022/2026,07947821000189,False,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30T14:37:39.646059


#### Também inclui o valor "PE", que parece se referir a instições internacionais

### `nomerazaosocialfornecedor`
Nome ou razão social do fornecedor

In [165]:
raw_resultado_itens_contratacoes.nomerazaosocialfornecedor.value_counts(1, dropna=False).head()

nomerazaosocialfornecedor
GOLDEN PRODUTOS ODONTOLOGICOS LTDA                                0.005592
SOUL DISTRIBUIDORA DE PRODUTOS E EQUIPAMENTOS INDUSTRIAIS LTDA    0.004066
RODRIZE MECANICA DE CAMINHOES LTDA                                0.003535
SUNERGY FARM AND CONSTRUCTION LTDA                                0.002961
CRISTALIA PRODUTOS QUIMICOS FARMACEUTICOS LTDA                    0.002919
Name: proportion, dtype: float64

In [166]:
raw_resultado_itens_contratacoes.nomerazaosocialfornecedor.value_counts(0, dropna=False).tail()

nomerazaosocialfornecedor
HTH LOCACAO E CONSULTORIA LTDA                                                      1
MP ENGENHARIA LTDA                                                                  1
GBOX GUARDA MOVEIS E LOCACOES LTDA                                                  1
COSTA MULLER CLIMATIZACAO LTDA                                                      1
SURGIMED COMERCIO ATACADISTA DE PRODUTOS CIRURGICOS E HOSPITALAR DO NORDESTE LTD    1
Name: count, dtype: int64

In [167]:
print((raw_resultado_itens_contratacoes.nomerazaosocialfornecedor.value_counts(1, dropna=False).head(10).round(4)*100).to_markdown())

| nomerazaosocialfornecedor                                                       |   proportion |
|:--------------------------------------------------------------------------------|-------------:|
| GOLDEN PRODUTOS ODONTOLOGICOS LTDA                                              |         0.56 |
| SOUL DISTRIBUIDORA DE PRODUTOS E EQUIPAMENTOS INDUSTRIAIS LTDA                  |         0.41 |
| RODRIZE MECANICA DE CAMINHOES LTDA                                              |         0.35 |
| SUNERGY FARM AND CONSTRUCTION LTDA                                              |         0.3  |
| CRISTALIA PRODUTOS QUIMICOS FARMACEUTICOS LTDA                                  |         0.29 |
| NUTRICELLI COMERCIO DE ALIMENTOS LTDA                                           |         0.29 |
| SUPREMA DENTAL IMPORTACAO, EXPORTACAO E COMERCIO DE PRODUTOS ODONTOLOGICOS LTDA |         0.29 |
| KLAS SUPRIMENTOS LTDA                                                           |         0.26 |
| DISTRIBU

#### Top 10 Fornecedores na base
| nomerazaosocialfornecedor                                                       |   Proporção |
|:--------------------------------------------------------------------------------|-------------:|
| GOLDEN PRODUTOS ODONTOLOGICOS LTDA                                              |         0,56 % |
| SOUL DISTRIBUIDORA DE PRODUTOS E EQUIPAMENTOS INDUSTRIAIS LTDA                  |         0,41 % |
| RODRIZE MECANICA DE CAMINHOES LTDA                                              |         0,35 % |
| SUNERGY FARM AND CONSTRUCTION LTDA                                              |         0,3  % |
| CRISTALIA PRODUTOS QUIMICOS FARMACEUTICOS LTDA                                  |         0,29 % |
| NUTRICELLI COMERCIO DE ALIMENTOS LTDA                                           |         0,29 % |
| SUPREMA DENTAL IMPORTACAO, EXPORTACAO E COMERCIO DE PRODUTOS ODONTOLOGICOS LTDA |         0,29 % |
| KLAS SUPRIMENTOS LTDA                                                           |         0,26 % |
| DISTRIBUIDORA MATHEUS LTDA                                                      |         0,24 % |
| VAMIX DISTRIBUIDORA LTDA                                                        |         0,24 % |

### `codigopais`
Código do país do fornecedor

In [168]:
raw_resultado_itens_contratacoes.codigopais.value_counts(1, dropna=False).head()

codigopais
BRA    0.998114
USA    0.000886
FRA    0.000192
ESP    0.000141
NLD    0.000108
Name: proportion, dtype: float64

In [169]:
raw_resultado_itens_contratacoes[raw_resultado_itens_contratacoes['codigopais'] != "BRA"].codigopais.value_counts(1, dropna=False).head()

codigopais
USA    0.469745
FRA    0.101911
ESP    0.074841
NLD    0.057325
DEU    0.057325
Name: proportion, dtype: float64

In [170]:
print((raw_resultado_itens_contratacoes[raw_resultado_itens_contratacoes['codigopais'] != "BRA"].codigopais.value_counts(1, dropna=False).head(10).round(4)*100).to_markdown())

| codigopais   |   proportion |
|:-------------|-------------:|
| USA          |        46.97 |
| FRA          |        10.19 |
| ESP          |         7.48 |
| NLD          |         5.73 |
| DEU          |         5.73 |
| CHE          |         3.34 |
| CAN          |         3.03 |
| PRT          |         2.71 |
| GBR          |         2.23 |
| NOR          |         1.91 |

#### Top 10 Fornecedores Estrangeiros na base
| codigopais   |   Proporção |
|:-------------|-------------:|
| USA          |        46.97 |
| FRA          |        10.19 |
| ESP          |         7.48 |
| NLD          |         5.73 |
| DEU          |         5.73 |
| CHE          |         3.34 |
| CAN          |         3.03 |
| PRT          |         2.71 |
| GBR          |         2.23 |
| NOR          |         1.91 |

### `indicadorsubcontratacao`
Indicador se há subcontratação (0 – False/Não, 1 – True/Sim)

In [171]:
raw_resultado_itens_contratacoes.indicadorsubcontratacao.value_counts(0, dropna=False)

indicadorsubcontratacao
False    332986
True          1
Name: count, dtype: int64

#### Apenas um valor verdadeiro

In [172]:
raw_resultado_itens_contratacoes[raw_resultado_itens_contratacoes['indicadorsubcontratacao'] == True]

,idcompraitem,idcompra,idcontratacaopncp,unidadeorgaocodigounidade,unidadeorgaoufsigla,numeroitempncp,sequencialresultado,nifornecedor,tipopessoa,nomerazaosocialfornecedor,codigopais,indicadorsubcontratacao,ordemclassificacaosrp,quantidadehomologada,valorunitariohomologado,valortotalhomologado,percentualdesconto,situacaocompraitemresultadoid,situacaocompraitemresultadonome,motivocancelamento,portefornecedorid,portefornecedornome,naturezajuridicanome,naturezajuridicaid,datainclusaopncp,dataatualizacaopncp,datacancelamentopncp,dataresultadopncp,numerocontrolepncpcompra,orgaoentidadecnpj,aplicacaomargempreferencia,amparolegalmargempreferenciaid,amparolegalmargempreferencianome,aplicacaobeneficiomeepp,aplicacaocriteriodesempate,amparolegalcriteriodesempateid,amparolegalcriteriodesempatenome,moedaestrangeiraid,datacotacaomoedaestrangeira,valornominalmoedaestrangeira,paisorigemprodutoservicoid,timezonecotacaomoedaestrangeira,dt_ingest
128969,9273310390049202500001,92733103900492025,08892295000160-1-000055/2025,927331,MA,1,1,69388361000153,PJ,QUALITECH ENGENHARIA LTDA.,BRA,True,NaN,1.0,2119327.24,2119327.24,0.0,1,Informado,NaN,3,Demais,Sociedade Empresária Limitada,2062,2026-01-30T11:05:10,2026-01-30T11:05:10,NaN,2026-01-30T00:00:00,08892295000160-1-000055/2025,08892295000160,False,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30T13:54:02.798686


### `ordemclassificacaosrp`
Ordem de classificação no Sistema de Registro de Preços (SRP)

In [173]:
raw_resultado_itens_contratacoes.ordemclassificacaosrp.value_counts(1, dropna=False)

ordemclassificacaosrp
1.0    0.725938
NaN    0.274062
Name: proportion, dtype: float64

#### 27% das linhas vazias, todas as outras tem valor `1`

### `quantidadehomologada`
Quantidade homologada

In [174]:
raw_resultado_itens_contratacoes.quantidadehomologada.value_counts(1, dropna=False).head()

quantidadehomologada
1.0      0.110890
10.0     0.038902
2.0      0.037710
20.0     0.031130
100.0    0.030713
Name: proportion, dtype: float64

In [175]:
raw_resultado_itens_contratacoes.quantidadehomologada.value_counts(0, dropna=False).tail()

quantidadehomologada
2839.0     1
19682.0    1
5158.0     1
5886.0     1
2293.0     1
Name: count, dtype: int64

### `valorunitariohomologado`
Valor unitário homologado

In [176]:
raw_resultado_itens_contratacoes.valorunitariohomologado.value_counts(1, dropna=False).head()

valorunitariohomologado
60.0    0.004589
1.0     0.004451
5.0     0.003454
15.0    0.003291
10.0    0.003222
Name: proportion, dtype: float64

### `valortotalhomologado`
Valor total homologado

In [177]:
raw_resultado_itens_contratacoes.valortotalhomologado.value_counts(1, dropna=False).head()

valortotalhomologado
174000.00    0.002940
161769.36    0.002520
162000.00    0.002387
0.00         0.002345
6000.00      0.002201
Name: proportion, dtype: float64

#### (TO DO) 0,23% de valores iguais a `0`

### `percentualdesconto`
Percentual de desconto oferecido

In [178]:
raw_resultado_itens_contratacoes.percentualdesconto.value_counts(1, dropna=False).head()

percentualdesconto
0.00     0.986669
0.01     0.000676
25.00    0.000592
30.00    0.000438
20.00    0.000372
Name: proportion, dtype: float64

#### 98,6% tem desconto igual a 0

### `situacaocompraitemresultadoid`
Identificador da situação do item do resultado de compra

In [179]:
raw_resultado_itens_contratacoes.situacaocompraitemresultadoid.value_counts(1, dropna=False).head()

situacaocompraitemresultadoid
1    0.984171
2    0.015829
Name: proportion, dtype: float64

### `situacaocompraitemresultadonome`
Nome da situação do item do resultado de compra

In [180]:
raw_resultado_itens_contratacoes.situacaocompraitemresultadonome.value_counts(1, dropna=False).head()

situacaocompraitemresultadonome
Informado    0.984171
Cancelado    0.015829
Name: proportion, dtype: float64

#### 1,58% Cancelados

### `motivocancelamento`
Motivo do cancelamento

In [181]:
raw_resultado_itens_contratacoes.motivocancelamento.value_counts(0, dropna=True).head()

motivocancelamento
A pedido do licitante                                                                                                                                                                                                                              193
Após a homologação do certame e antes da formalização contratual, a equipe técnica interna desta instituição identificou a necessidade de revisão dos quantitativos estimados constantes do Estudo Técnico Preliminar e do Termo de Referência.    185
Falta de comprovação fiscal exigida no item 9.34.1.2 do Edital. A empresa não apresentou Notas Fiscais Eletrônicas para lastrear seu atestado de capacidade técnica.                                                                               169
Apresentação de desistência por parte da empresa vencedora, Lagos Vita, conforme documentos nos autos.                                                                                                                                          

In [182]:
raw_resultado_itens_contratacoes.motivocancelamento.value_counts(0, dropna=True).tail()

motivocancelamento
Ausência de abertura de conta no banco Bradesco.                                                      1
FORNECEDOR DESISTIU DO ITEM                                                                           1
Retroagir para ajuste do valor na plataforma da proposta nos termos apresentado                       1
A revogação foi efetuada por interesse da Administração Pública, nos termos da Lei nº 14.133/2021.    1
Retorno de Fase                                                                                       1
Name: count, dtype: int64

#### (TO DO) considerar possíveis valores categoricos

### `portefornecedorid`
Identificador do porte do fornecedor

In [183]:
raw_resultado_itens_contratacoes.portefornecedorid.value_counts(0, dropna=True)

portefornecedorid
1    260262
3     61930
5     10473
2       201
4       121
Name: count, dtype: int64

### `portefornecedornome`
Nome do porte do fornecedor

In [184]:
raw_resultado_itens_contratacoes.portefornecedornome.value_counts(1, dropna=True)

portefornecedornome
ME               0.781598
Demais           0.185983
Não Informado    0.031452
EPP              0.000604
Não se aplica    0.000363
Name: proportion, dtype: float64

#### Porte dos fornecedores
| portefornecedornome   |   proportion |
|:----------------------|-------------:|
| ME                    |        78.16 |
| Demais                |        18.6  |
| Não Informado         |         3.15 |
| EPP                   |         0.06 |
| Não se aplica         |         0.04 |

#### (TO DO) Investigar `Não Informado` e `Não se aplica`

### `naturezajuridicanome`
Nome da natureza jurídica do fornecedor

In [185]:
raw_resultado_itens_contratacoes.naturezajuridicanome.value_counts(1, dropna=True).head(10)

naturezajuridicanome
Sociedade Empresária Limitada    0.844153
Empresário (Individual)          0.108488
Sociedade Anônima Fechada        0.016724
Associação Privada               0.008074
Cooperativa                      0.006368
Sociedade Anônima Aberta         0.004608
Sociedade Simples Limitada       0.003143
Fundação Privada                 0.002183
Empresa Pública                  0.001931
Sociedade de Economia Mista      0.000815
Name: proportion, dtype: float64

In [186]:
raw_resultado_itens_contratacoes.naturezajuridicanome.value_counts(0, dropna=True).tail()

naturezajuridicanome
Consórcio Público de Direito Privado             1
Representação Diplomática Estrangeira            1
Clube/Fundo de Investimento                      1
Fundação Pública de Direito Público Federal      1
Fundação Pública de Direito Público Municipal    1
Name: count, dtype: int64

In [187]:
print((raw_resultado_itens_contratacoes.naturezajuridicanome.value_counts(1, dropna=False).head(10).round(4)*100).to_markdown())

| naturezajuridicanome          |   proportion |
|:------------------------------|-------------:|
| Sociedade Empresária Limitada |        81.54 |
| Empresário (Individual)       |        10.48 |
| nan                           |         3.41 |
| Sociedade Anônima Fechada     |         1.62 |
| Associação Privada            |         0.78 |
| Cooperativa                   |         0.62 |
| Sociedade Anônima Aberta      |         0.45 |
| Sociedade Simples Limitada    |         0.3  |
| Fundação Privada              |         0.21 |
| Empresa Pública               |         0.19 |

#### Top 10 Naturezas Jurídicas de fornecedores
| naturezajuridicanome          |   proportion |
|:------------------------------|-------------:|
| Sociedade Empresária Limitada |        81.54 |
| Empresário (Individual)       |        10.48 |
| nan                           |         3.41 |
| Sociedade Anônima Fechada     |         1.62 |
| Associação Privada            |         0.78 |
| Cooperativa                   |         0.62 |
| Sociedade Anônima Aberta      |         0.45 |
| Sociedade Simples Limitada    |         0.3  |
| Fundação Privada              |         0.21 |
| Empresa Pública               |         0.19 |

#### 3% de valores vazios

### `naturezajuridicaid`
Identificador da natureza jurídica do fornecedor

In [188]:
raw_resultado_itens_contratacoes.naturezajuridicaid.value_counts(1, dropna=True).head()

naturezajuridicaid
2062    0.844153
2135    0.108488
2054    0.016724
3999    0.008074
2143    0.006368
Name: proportion, dtype: float64

In [189]:
raw_resultado_itens_contratacoes.naturezajuridicaid.value_counts(0, dropna=True).tail()

naturezajuridicaid
1228    1
5029    1
2224    1
1139    1
1155    1
Name: count, dtype: int64

### `datainclusaopncp`
Data de inclusão do item no PNCP

In [190]:
raw_resultado_itens_contratacoes.datainclusaopncp.value_counts(0, dropna=True).head()

datainclusaopncp
2026-01-15T15:14:20    53
2026-01-15T15:14:22    51
2026-01-15T15:14:21    48
2026-01-15T15:14:19    44
2026-02-17T14:51:12    42
Name: count, dtype: int64

In [191]:
raw_resultado_itens_contratacoes.datainclusaopncp.value_counts(0, dropna=True).tail()

datainclusaopncp
2026-03-18T10:11:28    1
2026-03-18T10:11:29    1
2026-03-18T10:11:31    1
2026-03-18T12:56:15    1
2026-03-23T10:06:32    1
Name: count, dtype: int64

### `dataatualizacaopncp`
Data da última atualização do item no PNCP

In [192]:
raw_resultado_itens_contratacoes.dataatualizacaopncp.value_counts(0, dropna=True).head()

dataatualizacaopncp
2026-01-15T15:14:20    53
2026-01-15T15:14:22    51
2026-01-15T15:14:21    48
2026-01-15T15:14:19    44
2026-02-17T14:51:12    42
Name: count, dtype: int64

### `datacancelamentopncp`
Data de cancelamento do item no PNCP

In [193]:
raw_resultado_itens_contratacoes.datacancelamentopncp.value_counts(0, dropna=True).head()

datacancelamentopncp
2026-03-17T00:00:00    270
2026-03-25T00:00:00    227
2026-03-30T00:00:00    214
2026-04-13T00:00:00    182
2026-03-23T00:00:00    126
Name: count, dtype: int64

### `dataresultadopncp`
Data do resultado para o item no PNCP

In [194]:
raw_resultado_itens_contratacoes.dataresultadopncp.value_counts(0, dropna=True).head()

dataresultadopncp
2026-02-19T00:00:00    9335
2026-02-05T00:00:00    8120
2026-03-16T00:00:00    8052
2026-01-23T00:00:00    7810
2026-03-10T00:00:00    7645
Name: count, dtype: int64

### `numerocontrolepncpcompra`
Número de controle da compra no PNCP (não documentado explicitamente para esta tabela na API)

In [195]:
raw_resultado_itens_contratacoes.numerocontrolepncpcompra.value_counts(0, dropna=True).head()

numerocontrolepncpcompra
00394429000100-1-000524/2025    2061
02334933000140-1-000001/2026    1873
76995323000124-1-000245/2025    1341
00662270000168-1-000164/2025    1184
24416174000106-1-000126/2025     682
Name: count, dtype: int64

### `orgaoentidadecnpj`
Número do CNPJ da entidade do órgão

In [196]:
raw_resultado_itens_contratacoes.orgaoentidadecnpj.value_counts(0, dropna=True).head()

orgaoentidadecnpj
00394452000103    43776
15126437000305    13887
00394429000100    12047
00394502000144     9480
96291141000180     8492
Name: count, dtype: int64

In [197]:
raw_resultado_itens_contratacoes.orgaoentidadecnpj.value_counts(0, dropna=True).tail()

orgaoentidadecnpj
78113834000109    1
00237222000122    1
03491063000186    1
11283607000142    1
32538167000105    1
Name: count, dtype: int64

In [198]:
raw_resultado_itens_contratacoes['orgaoentidadecnpj'].str.len().value_counts()

orgaoentidadecnpj
14    332987
Name: count, dtype: int64

#### Todos cnpjs possuem 14 caracteres.

In [199]:
len(raw_resultado_itens_contratacoes['orgaoentidadecnpj'].unique())

2477

#### 2477 cnpjs de entidades de orgãos distintos

### `aplicacaomargempreferencia`
Indica se foi aplicada a margem de preferência na contratação (0 – False/Não, 1 – True/Sim)

In [200]:
raw_resultado_itens_contratacoes.aplicacaomargempreferencia.value_counts(0, dropna=False)

aplicacaomargempreferencia
False    332853
True        134
Name: count, dtype: int64

In [201]:
raw_resultado_itens_contratacoes.aplicacaomargempreferencia.value_counts(1, dropna=False)

aplicacaomargempreferencia
False    0.999598
True     0.000402
Name: proportion, dtype: float64

#### Apenas 134 com aplicação de margem de preferência (0,04%)

### `amparolegalmargempreferenciaid`
Identificador do amparo legal da margem de preferência aplicada

In [202]:
raw_resultado_itens_contratacoes.amparolegalmargempreferenciaid.value_counts(0, dropna=False)

amparolegalmargempreferenciaid
NaN      332853
143.0       126
144.0         8
Name: count, dtype: int64

#### Valores vazios se referem as linhas sem margem de preferência

### `amparolegalmargempreferencianome`
Nome do amparo legal da margem de preferência aplicada

In [203]:
raw_resultado_itens_contratacoes.amparolegalmargempreferencianome.value_counts(0, dropna=False)

amparolegalmargempreferencianome
NaN                                  332853
Lei 14.133/2021, art. 26, §1º, II       126
Lei 14.133/2021, art. 26, §2º             8
Name: count, dtype: int64

### `aplicacaobeneficiomeepp`
Indica se foi concedido benefício específico para ME/EPP (Micro e Pequenas Empresas)

0 – False/Não

1 – True/Sim

In [204]:
raw_resultado_itens_contratacoes.aplicacaobeneficiomeepp.value_counts(1, dropna=False)

aplicacaobeneficiomeepp
False    0.992621
True     0.007379
Name: proportion, dtype: float64

#### Benefício em apenas 0,7% das linhas

In [205]:
raw_resultado_itens_contratacoes.aplicacaobeneficiomeepp.value_counts(0, dropna=False)

aplicacaobeneficiomeepp
False    330530
True       2457
Name: count, dtype: int64

In [206]:
raw_resultado_itens_contratacoes[raw_resultado_itens_contratacoes['aplicacaobeneficiomeepp'] == True].portefornecedornome.value_counts(0, dropna=True)

portefornecedornome
ME     2456
EPP       1
Name: count, dtype: int64

### `aplicacaocriteriodesempate`
Indica se foi aplicado algum critério de desempate conforme a legislação

0 – False/Não

1 – True/Sim

In [207]:
raw_resultado_itens_contratacoes.aplicacaocriteriodesempate.value_counts(1, dropna=False)

aplicacaocriteriodesempate
False    0.981939
True     0.018061
Name: proportion, dtype: float64

#### Desempate em 1,8% das linhas

In [208]:
raw_resultado_itens_contratacoes.aplicacaocriteriodesempate.value_counts(0, dropna=False)

aplicacaocriteriodesempate
False    326973
True       6014
Name: count, dtype: int64

### `amparolegalcriteriodesempateid`
Identificador do amparo legal do critério de desempate aplicado

In [209]:
raw_resultado_itens_contratacoes.amparolegalcriteriodesempateid.value_counts(0, dropna=False)

amparolegalcriteriodesempateid
NaN      326973
148.0      2460
157.0      1950
163.0       647
162.0       340
145.0       320
146.0       297
Name: count, dtype: int64

In [210]:
raw_resultado_itens_contratacoes.amparolegalcriteriodesempateid.value_counts(1, dropna=False)

amparolegalcriteriodesempateid
NaN      0.981939
148.0    0.007388
157.0    0.005856
163.0    0.001943
162.0    0.001021
145.0    0.000961
146.0    0.000892
Name: proportion, dtype: float64

### `amparolegalcriteriodesempatenome`
Nome do amparo legal do critério de desempate aplicado

In [211]:
raw_resultado_itens_contratacoes.amparolegalcriteriodesempatenome.value_counts(1, dropna=False)

amparolegalcriteriodesempatenome
NaN                                  0.999598
Lei 14.133/2021, art. 26, §1º, II    0.000378
Lei 14.133/2021, art. 26, §2º        0.000024
Name: proportion, dtype: float64

#### (TO DO) Determinar quantos e quais ids de amparo legal não tem nome (poderia ser característica de uma fonte?)

### `moedaestrangeiraid`
Identificador da moeda estrangeira (quando aplicável)

In [212]:
raw_resultado_itens_contratacoes.moedaestrangeiraid.value_counts(0, dropna=False)

moedaestrangeiraid
NaN    332987
Name: count, dtype: int64

#### Coluna 100% vazia

### `datacotacaomoedaestrangeira`
Data da cotação da moeda estrangeira

In [213]:
raw_resultado_itens_contratacoes.datacotacaomoedaestrangeira.value_counts(0, dropna=False)

datacotacaomoedaestrangeira
NaN    332987
Name: count, dtype: int64

#### Coluna 100% vazia

### `valornominalmoedaestrangeira`
Valor nominal em moeda estrangeira

In [214]:
raw_resultado_itens_contratacoes.valornominalmoedaestrangeira.value_counts(0, dropna=False)

valornominalmoedaestrangeira
NaN    332987
Name: count, dtype: int64

#### Coluna 100% vazia

### `paisorigemprodutoservicoid`
Identificador do país de origem do produto ou serviço

In [215]:
raw_resultado_itens_contratacoes.paisorigemprodutoservicoid.value_counts(0, dropna=False)

paisorigemprodutoservicoid
NaN    332853
BRA       131
CHN         3
Name: count, dtype: int64

In [216]:
raw_resultado_itens_contratacoes.paisorigemprodutoservicoid.value_counts(1, dropna=False)

paisorigemprodutoservicoid
NaN    0.999598
BRA    0.000393
CHN    0.000009
Name: proportion, dtype: float64

#### 99% dos valorez vazios, tem apenas dois valores: `BRA` e `CHN`

#### (TO DO) Comparar com os valores da coluna do país da empresa fornecedora

### `timezonecotacaomoedaestrangeira`
Timezone da cotação da moeda estrangeira

In [217]:
raw_resultado_itens_contratacoes.timezonecotacaomoedaestrangeira.value_counts(0, dropna=False)

timezonecotacaomoedaestrangeira
NaN    332987
Name: count, dtype: int64

#### Coluna 100% vazia